## 1. Dataset Inventory

### Engineering Question

What files constitute the frozen measurement release, and are the expected
dataset artifacts available for qualification?

### Why This Check Matters

Before examining data quality or sensor behavior, the analysis must establish
which files belong to the measurement campaign and confirm that the intended
frozen release is being used.

This step establishes the input boundary for DQ-001 and prevents accidental
analysis of files outside the controlled dataset release.

In [108]:
from pathlib import Path
import json
import pandas as pd


# ------------------------------------------------------------
# Locate project root and frozen dataset release
# ------------------------------------------------------------

current_dir = Path.cwd()

candidate_roots = [
    current_dir,
    current_dir.parent
]

project_root = next(
    (
        path for path in candidate_roots
        if (path / "data" / "raw" / "vsm_lab_data_v1_1").exists()
    ),
    None
)

if project_root is None:
    raise FileNotFoundError(
        "Could not locate the project root containing "
        "'data/raw/vsm_lab_data_v1_1'."
    )

raw_data_dir = (
    project_root
    / "data"
    / "raw"
    / "vsm_lab_data_v1_1"
)

manifest_path = raw_data_dir / "dataset_manifest.json"
test_matrix_path = raw_data_dir / "test_matrix.csv"


print(f"Project root : {project_root}")
print(f"Dataset path : {raw_data_dir}")
print(f"Manifest     : {manifest_path.name}")
print(f"Test matrix  : {test_matrix_path.name}")

Project root : c:\MyDocument\hvac-sensor-validation-metrology
Dataset path : c:\MyDocument\hvac-sensor-validation-metrology\data\raw\vsm_lab_data_v1_1
Manifest     : dataset_manifest.json
Test matrix  : test_matrix.csv


### 1.1 Frozen Release Manifest

The dataset manifest defines the controlled analyst-facing release used in
this study. It records the release identity, expected files, row counts,
schemas, and cryptographic commitments.

At this stage, the manifest is used as the authoritative inventory for
dataset qualification. Sensor-performance conclusions are not evaluated here.

In [109]:
# ------------------------------------------------------------
# Load frozen release manifest
# ------------------------------------------------------------

with manifest_path.open("r", encoding="utf-8") as file:
    manifest = json.load(file)

print(f"Release ID      : {manifest['release_id']}")
print(f"Schema version  : {manifest['schema_version']}")
print(f"Campaign        : {manifest['lab']['campaign']}")
print(f"Measurand       : {manifest['lab']['measurand']}")
print(f"Units           : {manifest['lab']['units']}")
print(
    f"Private truth   : "
    f"{'INCLUDED' if manifest['analyst_private_truth_included'] else 'NOT INCLUDED'}"
)

Release ID      : VSM-LAB-DATA-v1.1
Schema version  : 1.0
Campaign        : HVAC Relative-Humidity Sensor Validation
Measurand       : relative_humidity
Units           : %RH
Private truth   : NOT INCLUDED


In [110]:
# ------------------------------------------------------------
# Build manifest-based dataset inventory
# ------------------------------------------------------------

inventory_records = []

for file_name, file_info in manifest["files"].items():
    inventory_records.append(
        {
            "File": file_name,
            "Rows": file_info["rows"],
            "Columns": len(file_info["columns"]),
            "SHA256": file_info["sha256"],
        }
    )

inventory_df = (
    pd.DataFrame(inventory_records)
    .sort_values("File")
    .reset_index(drop=True)
)

display(inventory_df)

,File,Rows,Columns,SHA256
0,cyclic_measurements.csv,9003,14,f9379d3119e7dfbb21301278907790d39204e14c8d31ce...
1,dynamic_step_measurements.csv,9030,13,1dd80275b7c80377c99cd0e4cb2110ed9aad8346b336fa...
2,ramp_measurements.csv,21618,13,79d168de922c15358029106f5d7553da9287ee39d8fc33...
3,sinusoidal_measurements.csv,5404,14,a3bcd74e3604d50aa047a58d5ac6bf6412d4b30d1f21bf...
4,stability_measurements.csv,1320,14,cd519d42a0f93a5defcd067908702ae907247fa726cefc...
5,static_measurements.csv,15120,15,3f34e8a9bc31a81e32700d7347f96947351770e1b2c208...
6,test_matrix.csv,333,31,51302b9ece317bf39759a21518bb67c3b93e65f9f9b2c7...


### 1.2 Expected vs Actual Release Files

The manifest defines the analytical data files expected in the frozen release.
The release package also contains supporting provenance artifacts such as the
manifest itself, the release commitment, and the release README.

This check confirms that the controlled raw-data directory contains the
expected release package before deeper qualification checks are performed.

In [111]:
# ------------------------------------------------------------
# Compare expected release files with actual directory contents
# ------------------------------------------------------------

manifest_files = set(manifest["files"].keys())

support_files = {
    "dataset_manifest.json",
    "release_commitment_v1_1.json",
    "README.md",
}

expected_release_files = manifest_files | support_files

actual_release_files = {
    path.name
    for path in raw_data_dir.iterdir()
    if path.is_file()
}

missing_files = sorted(expected_release_files - actual_release_files)
unexpected_files = sorted(actual_release_files - expected_release_files)

print(f"Expected files   : {len(expected_release_files)}")
print(f"Actual files     : {len(actual_release_files)}")
print(f"Missing files    : {len(missing_files)}")
print(f"Unexpected files : {len(unexpected_files)}")

if missing_files:
    print("\nMissing:")
    for file_name in missing_files:
        print(f"  - {file_name}")

if unexpected_files:
    print("\nUnexpected:")
    for file_name in unexpected_files:
        print(f"  - {file_name}")

inventory_status = (
    "PASS"
    if not missing_files and not unexpected_files
    else "REVIEW"
)

print(f"\nRelease inventory status: {inventory_status}")

Expected files   : 10
Actual files     : 10
Missing files    : 0
Unexpected files : 0

Release inventory status: PASS


## 2. Schema & Structural Integrity

### Engineering Question

Do the measurement and test-design files contain the expected columns and
record structure defined by the frozen release manifest?

### Why This Check Matters

A file can exist and still be unusable if its schema has changed, required
columns are missing, or unexpected columns have been introduced.

Before any sensor-performance analysis is performed, each analytical file
must be checked against the controlled release definition.

In [112]:
# ------------------------------------------------------------
# Load analytical CSV files and compare schemas with manifest
# ------------------------------------------------------------

datasets = {}
schema_results = []

for file_name, file_info in manifest["files"].items():
    file_path = raw_data_dir / file_name

    df = pd.read_csv(file_path)
    datasets[file_name] = df

    expected_columns = file_info["columns"]
    actual_columns = df.columns.tolist()

    missing_columns = sorted(
        set(expected_columns) - set(actual_columns)
    )

    unexpected_columns = sorted(
        set(actual_columns) - set(expected_columns)
    )

    order_matches = actual_columns == expected_columns

    schema_results.append(
        {
            "File": file_name,
            "Expected_Columns": len(expected_columns),
            "Actual_Columns": len(actual_columns),
            "Missing_Columns": len(missing_columns),
            "Unexpected_Columns": len(unexpected_columns),
            "Column_Order_Matches": order_matches,
            "Schema_Status": (
                "PASS"
                if (
                    not missing_columns
                    and not unexpected_columns
                    and order_matches
                )
                else "FAIL"
            ),
        }
    )

schema_df = pd.DataFrame(schema_results)

display(schema_df)

,File,Expected_Columns,Actual_Columns,Missing_Columns,Unexpected_Columns,Column_Order_Matches,Schema_Status
0,cyclic_measurements.csv,14,14,0,0,True,PASS
1,dynamic_step_measurements.csv,13,13,0,0,True,PASS
2,ramp_measurements.csv,13,13,0,0,True,PASS
3,sinusoidal_measurements.csv,14,14,0,0,True,PASS
4,stability_measurements.csv,14,14,0,0,True,PASS
5,static_measurements.csv,15,15,0,0,True,PASS
6,test_matrix.csv,31,31,0,0,True,PASS


In [113]:
# ------------------------------------------------------------
# Final schema qualification
# ------------------------------------------------------------

total_missing_columns = schema_df["Missing_Columns"].sum()
total_unexpected_columns = schema_df["Unexpected_Columns"].sum()
column_order_mismatches = (~schema_df["Column_Order_Matches"]).sum()

schema_pass = (
    total_missing_columns == 0
    and total_unexpected_columns == 0
    and column_order_mismatches == 0
)

print(f"Missing columns       : {total_missing_columns}")
print(f"Unexpected columns    : {total_unexpected_columns}")
print(f"Column order mismatch : {column_order_mismatches}")

print(
    f"\nSchema qualification  : "
    f"{'PASS' if schema_pass else 'FAIL'}"
)

Missing columns       : 0
Unexpected columns    : 0
Column order mismatch : 0

Schema qualification  : PASS


### 2.1 Row-Count Integrity

The frozen release manifest specifies the expected number of records for each
analytical file.

This check compares the observed number of rows in each loaded dataset with the
manifest commitment. A row-count mismatch may indicate missing records,
unexpected additions, or an incomplete dataset copy and must be resolved before
downstream analysis.

In [114]:
# ------------------------------------------------------------
# Compare observed row counts with manifest expectations
# ------------------------------------------------------------

row_count_results = []

for file_name, file_info in manifest["files"].items():

    expected_rows = file_info["rows"]
    observed_rows = len(datasets[file_name])

    row_count_results.append(
        {
            "File": file_name,
            "Expected_Rows": expected_rows,
            "Observed_Rows": observed_rows,
            "Difference": observed_rows - expected_rows,
            "Row_Count_Status": (
                "PASS"
                if observed_rows == expected_rows
                else "FAIL"
            ),
        }
    )

row_count_df = pd.DataFrame(row_count_results)

display(row_count_df)

,File,Expected_Rows,Observed_Rows,Difference,Row_Count_Status
0,cyclic_measurements.csv,9003,9003,0,PASS
1,dynamic_step_measurements.csv,9030,9030,0,PASS
2,ramp_measurements.csv,21618,21618,0,PASS
3,sinusoidal_measurements.csv,5404,5404,0,PASS
4,stability_measurements.csv,1320,1320,0,PASS
5,static_measurements.csv,15120,15120,0,PASS
6,test_matrix.csv,333,333,0,PASS


In [115]:
row_count_pass = (
    row_count_df["Row_Count_Status"] == "PASS"
).all()

total_row_difference = row_count_df["Difference"].abs().sum()

print(f"Total absolute row difference : {total_row_difference}")
print(
    f"Row-count qualification       : "
    f"{'PASS' if row_count_pass else 'FAIL'}"
)

Total absolute row difference : 0
Row-count qualification       : PASS


### 2.2 Missing-Value Audit

Missing values are evaluated only in fields that are required for the
corresponding measurement family.

Not every blank field represents a data-quality defect. In particular,
`test_matrix.csv` contains family-specific design parameters, so many fields
are intentionally not applicable to every test.

This check therefore distinguishes between:

- missing values in required measurement fields, which may indicate a defect
- structurally not-applicable fields, which are not automatically considered failures

In [116]:
# ------------------------------------------------------------
# Define required fields for each measurement dataset
# ------------------------------------------------------------

required_fields = {
    "static_measurements.csv": [
        "Elapsed_s",
        "Sample_ID",
        "Test_ID",
        "Test_Family",
        "Analysis_Role",
        "Day_ID",
        "Run_ID",
        "RH_Setpoint",
        "Temperature_Setpoint_C",
        "Direction",
        "Chamber_State",
        "RH_Reference",
        "RH_DUT",
        "Temperature_Reference_C",
        "Temperature_DUT_C",
    ],

    "dynamic_step_measurements.csv": [
        "Elapsed_s",
        "Sample_ID",
        "Test_ID",
        "Test_Family",
        "RH_Command",
        "Temperature_Setpoint_C",
        "Direction",
        "Chamber_State",
        "RH_Reference",
        "RH_DUT",
        "Temperature_Reference_C",
        "Temperature_DUT_C",
        "Repetition",
    ],

    "ramp_measurements.csv": [
        "Elapsed_s",
        "Sample_ID",
        "Test_ID",
        "Test_Family",
        "RH_Command",
        "Temperature_Setpoint_C",
        "Direction",
        "Chamber_State",
        "RH_Reference",
        "RH_DUT",
        "Temperature_Reference_C",
        "Temperature_DUT_C",
        "Repetition",
    ],

    "cyclic_measurements.csv": [
        "Elapsed_s",
        "Sample_ID",
        "Test_ID",
        "Test_Family",
        "RH_Command",
        "Temperature_Setpoint_C",
        "Direction",
        "Chamber_State",
        "RH_Reference",
        "RH_DUT",
        "Temperature_Reference_C",
        "Temperature_DUT_C",
        "Session",
        "Cycle_Index",
    ],

    "sinusoidal_measurements.csv": [
        "Elapsed_s",
        "Sample_ID",
        "Test_ID",
        "Test_Family",
        "RH_Command",
        "Temperature_Setpoint_C",
        "Direction",
        "Chamber_State",
        "RH_Reference",
        "RH_DUT",
        "Temperature_Reference_C",
        "Temperature_DUT_C",
        "Period_s",
        "Frequency_Hz",
    ],

    "stability_measurements.csv": [
        "Elapsed_s",
        "Sample_ID",
        "Test_ID",
        "Test_Family",
        "Schedule_Type",
        "Aging_Day",
        "RH_Setpoint",
        "Temperature_Setpoint_C",
        "Direction",
        "Chamber_State",
        "RH_Reference",
        "RH_DUT",
        "Temperature_Reference_C",
        "Temperature_DUT_C",
    ],
}

In [117]:
# ------------------------------------------------------------
# Audit missing values in required measurement fields
# ------------------------------------------------------------

missing_value_results = []

for file_name, columns in required_fields.items():

    df = datasets[file_name]

    missing_count = df[columns].isna().sum().sum()

    rows_with_missing = df[columns].isna().any(axis=1).sum()

    missing_value_results.append(
        {
            "File": file_name,
            "Required_Fields": len(columns),
            "Missing_Required_Values": int(missing_count),
            "Rows_With_Missing_Required_Values": int(rows_with_missing),
            "Missing_Value_Status": (
                "PASS"
                if missing_count == 0
                else "FAIL"
            ),
        }
    )

missing_value_df = pd.DataFrame(missing_value_results)

display(missing_value_df)

,File,Required_Fields,Missing_Required_Values,Rows_With_Missing_Required_Values,Missing_Value_Status
0,static_measurements.csv,15,0,0,PASS
1,dynamic_step_measurements.csv,13,0,0,PASS
2,ramp_measurements.csv,13,0,0,PASS
3,cyclic_measurements.csv,14,0,0,PASS
4,sinusoidal_measurements.csv,14,0,0,PASS
5,stability_measurements.csv,14,0,0,PASS


In [118]:
missing_value_pass = (
    missing_value_df["Missing_Value_Status"] == "PASS"
).all()

total_missing_required = (
    missing_value_df["Missing_Required_Values"].sum()
)

print(f"Missing required measurement values : {total_missing_required}")
print(
    f"Required-field completeness         : "
    f"{'PASS' if missing_value_pass else 'FAIL'}"
)

Missing required measurement values : 0
Required-field completeness         : PASS


### 2.3 Duplicate Integrity

Duplicate records can artificially increase the apparent amount of measurement
evidence and distort statistical summaries.

This audit distinguishes between:

- exact duplicate rows
- duplicate measurement identities within a test
- duplicate `Test_ID` entries in the experimental test matrix

Identical measured values are not automatically duplicates. The concern is
whether the same measurement record or experimental identifier has been
represented more than once.

In [119]:
# ------------------------------------------------------------
# Audit exact duplicates and measurement-key duplicates
# ------------------------------------------------------------

duplicate_results = []

measurement_files = [
    file_name
    for file_name in datasets
    if file_name != "test_matrix.csv"
]

for file_name in measurement_files:

    df = datasets[file_name]

    # Exact duplicated rows
    exact_duplicate_rows = int(
        df.duplicated(keep=False).sum()
    )

    # Measurement identity should be unique within each test
    duplicate_measurement_keys = int(
        df.duplicated(
            subset=["Test_ID", "Sample_ID"],
            keep=False
        ).sum()
    )

    duplicate_results.append(
        {
            "File": file_name,
            "Exact_Duplicate_Rows": exact_duplicate_rows,
            "Duplicate_TestID_SampleID": duplicate_measurement_keys,
            "Duplicate_Status": (
                "PASS"
                if (
                    exact_duplicate_rows == 0
                    and duplicate_measurement_keys == 0
                )
                else "FAIL"
            ),
        }
    )

duplicate_df = pd.DataFrame(duplicate_results)

display(duplicate_df)

,File,Exact_Duplicate_Rows,Duplicate_TestID_SampleID,Duplicate_Status
0,cyclic_measurements.csv,0,0,PASS
1,dynamic_step_measurements.csv,0,0,PASS
2,ramp_measurements.csv,0,0,PASS
3,sinusoidal_measurements.csv,0,0,PASS
4,stability_measurements.csv,0,0,PASS
5,static_measurements.csv,0,0,PASS


In [120]:
# ------------------------------------------------------------
# Check Test_ID uniqueness in the experimental test matrix
# ------------------------------------------------------------

test_matrix_df = datasets["test_matrix.csv"]

duplicate_test_ids = int(
    test_matrix_df.duplicated(
        subset=["Test_ID"],
        keep=False
    ).sum()
)

print(f"Duplicate Test_ID rows in test matrix : {duplicate_test_ids}")

Duplicate Test_ID rows in test matrix : 0


In [121]:
measurement_duplicate_pass = (
    duplicate_df["Duplicate_Status"] == "PASS"
).all()

test_matrix_duplicate_pass = (
    duplicate_test_ids == 0
)

duplicate_integrity_pass = (
    measurement_duplicate_pass
    and test_matrix_duplicate_pass
)

total_exact_duplicates = (
    duplicate_df["Exact_Duplicate_Rows"].sum()
)

total_duplicate_keys = (
    duplicate_df["Duplicate_TestID_SampleID"].sum()
)

print(f"Exact duplicate measurement rows       : {total_exact_duplicates}")
print(f"Duplicate Test_ID/Sample_ID records    : {total_duplicate_keys}")
print(f"Duplicate Test_ID rows in test matrix  : {duplicate_test_ids}")

print(
    f"\nDuplicate integrity qualification      : "
    f"{'PASS' if duplicate_integrity_pass else 'FAIL'}"
)

Exact duplicate measurement rows       : 0
Duplicate Test_ID/Sample_ID records    : 0
Duplicate Test_ID rows in test matrix  : 0

Duplicate integrity qualification      : PASS


## 3. Test-ID Traceability

### Engineering Question

Can every measurement record be traced to a valid experimental test defined
in `test_matrix.csv`?

### Why This Check Matters

Each measurement must belong to a known test condition and experimental
purpose. If a measurement contains an unknown `Test_ID`, its origin and test
conditions cannot be established reliably.

This check verifies that all measurement-producing `Test_ID` values are
represented in the controlled experimental test matrix.

In [122]:
# ------------------------------------------------------------
# Collect Test_ID values from all measurement files
# ------------------------------------------------------------

measurement_test_ids = set()

test_id_summary = []

for file_name in measurement_files:

    df = datasets[file_name]

    file_test_ids = set(df["Test_ID"].dropna().unique())

    measurement_test_ids.update(file_test_ids)

    test_id_summary.append(
        {
            "File": file_name,
            "Unique_Test_IDs": len(file_test_ids),
        }
    )

test_id_summary_df = pd.DataFrame(test_id_summary)

display(test_id_summary_df)

print(
    f"Total unique measurement Test_IDs : "
    f"{len(measurement_test_ids)}"
)

,File,Unique_Test_IDs
0,cyclic_measurements.csv,3
1,dynamic_step_measurements.csv,30
2,ramp_measurements.csv,18
3,sinusoidal_measurements.csv,4
4,stability_measurements.csv,22
5,static_measurements.csv,252


Total unique measurement Test_IDs : 329


In [123]:
# ------------------------------------------------------------
# Compare measurement Test_IDs with the test matrix
# ------------------------------------------------------------

matrix_test_ids = set(
    test_matrix_df["Test_ID"]
    .dropna()
    .unique()
)

measurement_ids_not_in_matrix = sorted(
    measurement_test_ids - matrix_test_ids
)

matrix_ids_without_measurements = sorted(
    matrix_test_ids - measurement_test_ids
)

print(f"Test matrix Test_IDs                 : {len(matrix_test_ids)}")
print(f"Measurement Test_IDs                 : {len(measurement_test_ids)}")
print(
    f"Measurement IDs missing from matrix  : "
    f"{len(measurement_ids_not_in_matrix)}"
)
print(
    f"Matrix IDs without measurement rows  : "
    f"{len(matrix_ids_without_measurements)}"
)

Test matrix Test_IDs                 : 333
Measurement Test_IDs                 : 329
Measurement IDs missing from matrix  : 0
Matrix IDs without measurement rows  : 4


In [124]:
# ------------------------------------------------------------
# Classify test-matrix entries without measurement rows
# ------------------------------------------------------------

unmatched_matrix_rows = test_matrix_df[
    test_matrix_df["Test_ID"].isin(
        matrix_ids_without_measurements
    )
][
    ["Test_ID", "Test_Family", "Point_Type"]
]

display(unmatched_matrix_rows)

,Test_ID,Test_Family,Point_Type
329,UNC-001,UNCERTAINTY_EVALUATION,REPRESENTATIVE
330,UNC-002,UNCERTAINTY_EVALUATION,REPRESENTATIVE
331,UNC-003,UNCERTAINTY_EVALUATION,REPRESENTATIVE
332,UNC-004,UNCERTAINTY_EVALUATION,WORST_CASE


In [125]:
# ------------------------------------------------------------
# Evaluate Test-ID traceability
# ------------------------------------------------------------

unexpected_unmatched_matrix_rows = unmatched_matrix_rows[
    unmatched_matrix_rows["Test_Family"]
    != "UNCERTAINTY_EVALUATION"
]

test_id_traceability_pass = (
    len(measurement_ids_not_in_matrix) == 0
    and len(unexpected_unmatched_matrix_rows) == 0
)

print(
    f"Untraceable measurement Test_IDs : "
    f"{len(measurement_ids_not_in_matrix)}"
)

print(
    f"Expected non-measurement design points : "
    f"{len(unmatched_matrix_rows)}"
)

print(
    f"Unexpected matrix-only Test_IDs   : "
    f"{len(unexpected_unmatched_matrix_rows)}"
)

print(
    f"\nTest-ID traceability qualification : "
    f"{'PASS' if test_id_traceability_pass else 'FAIL'}"
)

Untraceable measurement Test_IDs : 0
Expected non-measurement design points : 4
Unexpected matrix-only Test_IDs   : 0

Test-ID traceability qualification : PASS


## 4. Test Completeness

### Engineering Question

Do the measurement-producing tests contain the expected amount of measurement
evidence defined by the experimental design?

### Why This Check Matters

A dataset may have the correct total number of rows while still containing an
incomplete individual test.

Test-level completeness is therefore evaluated separately from file-level
row-count integrity.

Where the test matrix explicitly defines `Expected_Samples`, the observed
number of measurement records is compared directly with that expectation.

Tests for which an explicit expected sample count is not defined are retained
for separate family-specific qualification rather than being automatically
classified as failures.

In [126]:
# ------------------------------------------------------------
# Count observed measurement records for each Test_ID
# ------------------------------------------------------------

observed_test_counts = []

for file_name in measurement_files:

    df = datasets[file_name]

    counts = (
        df.groupby("Test_ID")
        .size()
        .reset_index(name="Observed_Samples")
    )

    counts["Source_File"] = file_name

    observed_test_counts.append(counts)

observed_test_counts_df = pd.concat(
    observed_test_counts,
    ignore_index=True
)

display(observed_test_counts_df.head(10))

print(
    f"Measurement-producing Test_IDs : "
    f"{observed_test_counts_df['Test_ID'].nunique()}"
)

,Test_ID,Observed_Samples,Source_File
0,DY-CYC-001,3001,cyclic_measurements.csv
1,DY-CYC-002,3001,cyclic_measurements.csv
2,DY-CYC-003,3001,cyclic_measurements.csv
3,DY-STEP-001,301,dynamic_step_measurements.csv
4,DY-STEP-002,301,dynamic_step_measurements.csv
5,DY-STEP-003,301,dynamic_step_measurements.csv
6,DY-STEP-004,301,dynamic_step_measurements.csv
7,DY-STEP-005,301,dynamic_step_measurements.csv
8,DY-STEP-006,301,dynamic_step_measurements.csv
9,DY-STEP-007,301,dynamic_step_measurements.csv


Measurement-producing Test_IDs : 329


In [127]:
# ------------------------------------------------------------
# Join observed sample counts to the experimental test matrix
# ------------------------------------------------------------

test_completeness_df = test_matrix_df.merge(
    observed_test_counts_df,
    on="Test_ID",
    how="left"
)

display(
    test_completeness_df[
        [
            "Test_ID",
            "Test_Family",
            "Expected_Samples",
            "Observed_Samples",
            "Source_File",
        ]
    ].head(10)
)

,Test_ID,Test_Family,Expected_Samples,Observed_Samples,Source_File
0,CAL-001,STATIC,60.0,60.0,static_measurements.csv
1,CAL-002,STATIC,60.0,60.0,static_measurements.csv
2,CAL-003,STATIC,60.0,60.0,static_measurements.csv
3,CAL-004,STATIC,60.0,60.0,static_measurements.csv
4,ST-001,STATIC,60.0,60.0,static_measurements.csv
5,ST-002,STATIC,60.0,60.0,static_measurements.csv
6,CAL-005,STATIC,60.0,60.0,static_measurements.csv
7,CAL-006,STATIC,60.0,60.0,static_measurements.csv
8,CAL-007,STATIC,60.0,60.0,static_measurements.csv
9,CAL-008,STATIC,60.0,60.0,static_measurements.csv


In [128]:
expected_sample_coverage = (
    test_completeness_df
    .groupby("Test_Family", dropna=False)
    .agg(
        Total_Tests=("Test_ID", "size"),
        Tests_With_Expected_Samples=(
            "Expected_Samples",
            lambda x: x.notna().sum()
        ),
        Tests_With_Measurements=(
            "Observed_Samples",
            lambda x: x.notna().sum()
        ),
    )
    .reset_index()
)

expected_sample_coverage[
    "Tests_Without_Expected_Samples"
] = (
    expected_sample_coverage["Total_Tests"]
    - expected_sample_coverage["Tests_With_Expected_Samples"]
)

display(expected_sample_coverage)

,Test_Family,Total_Tests,Tests_With_Expected_Samples,Tests_With_Measurements,Tests_Without_Expected_Samples
0,DYNAMIC_CYCLIC,3,0,3,3
1,DYNAMIC_RAMP,18,0,18,18
2,DYNAMIC_SINUSOIDAL,4,0,4,4
3,DYNAMIC_STEP,30,0,30,30
4,STABILITY,22,0,22,22
5,STATIC,252,252,252,0
6,UNCERTAINTY_EVALUATION,4,0,0,4


### 4.4 Family-Aware Completeness Logic

Sample completeness cannot be evaluated using a single rule for all test
families because the experimental design represents different tests in
different ways.

The qualification logic is therefore family-aware:

- `STATIC`: use the explicit `Expected_Samples` value.
- `DYNAMIC_STEP`: derive the expected count from `Duration_s` and
  `Sample_Rate_Hz`, including the sample at time zero.
- `DYNAMIC_RAMP`: derive the expected count from `Duration_s` and
  `Sample_Rate_Hz`, including the sample at time zero.
- `DYNAMIC_CYCLIC`, `DYNAMIC_SINUSOIDAL`, and `STABILITY`: defer strict
  completeness assessment to family-specific temporal and sequence checks.
- `UNCERTAINTY_EVALUATION`: design-only points; standalone measurement rows
  are not expected.

This prevents structurally valid test-family differences from being
incorrectly classified as missing data.

In [129]:
# ------------------------------------------------------------
# Build family-aware expected sample counts
# ------------------------------------------------------------

test_completeness_df["Derived_Expected_Samples"] = (
    test_completeness_df["Expected_Samples"]
)

# Dynamic tests with explicit duration and sample rate
duration_based_mask = (
    test_completeness_df["Expected_Samples"].isna()
    & test_completeness_df["Duration_s"].notna()
    & test_completeness_df["Sample_Rate_Hz"].notna()
)

test_completeness_df.loc[
    duration_based_mask,
    "Derived_Expected_Samples"
] = (
    test_completeness_df.loc[
        duration_based_mask,
        "Duration_s"
    ]
    *
    test_completeness_df.loc[
        duration_based_mask,
        "Sample_Rate_Hz"
    ]
    + 1
)

display(
    test_completeness_df[
        [
            "Test_ID",
            "Test_Family",
            "Expected_Samples",
            "Duration_s",
            "Sample_Rate_Hz",
            "Derived_Expected_Samples",
            "Observed_Samples",
        ]
    ].query(
        "Test_Family in ['STATIC', 'DYNAMIC_STEP', 'DYNAMIC_RAMP']"
    ).head(15)
)

,Test_ID,Test_Family,Expected_Samples,Duration_s,Sample_Rate_Hz,Derived_Expected_Samples,Observed_Samples
0,CAL-001,STATIC,60.0,NaN,1.0,60.0,60.0
1,CAL-002,STATIC,60.0,NaN,1.0,60.0,60.0
2,CAL-003,STATIC,60.0,NaN,1.0,60.0,60.0
3,CAL-004,STATIC,60.0,NaN,1.0,60.0,60.0
4,ST-001,STATIC,60.0,NaN,1.0,60.0,60.0
5,ST-002,STATIC,60.0,NaN,1.0,60.0,60.0
6,CAL-005,STATIC,60.0,NaN,1.0,60.0,60.0
7,CAL-006,STATIC,60.0,NaN,1.0,60.0,60.0
8,CAL-007,STATIC,60.0,NaN,1.0,60.0,60.0
9,CAL-008,STATIC,60.0,NaN,1.0,60.0,60.0


In [130]:
display(
    test_completeness_df[
        [
            "Test_ID",
            "Test_Family",
            "Duration_s",
            "Sample_Rate_Hz",
            "Derived_Expected_Samples",
            "Observed_Samples",
        ]
    ].query(
        "Test_Family in ['DYNAMIC_STEP', 'DYNAMIC_RAMP']"
    ).head(10)
)

,Test_ID,Test_Family,Duration_s,Sample_Rate_Hz,Derived_Expected_Samples,Observed_Samples
252,DY-STEP-001,DYNAMIC_STEP,300.0,1.0,301.0,301.0
253,DY-STEP-002,DYNAMIC_STEP,300.0,1.0,301.0,301.0
254,DY-STEP-003,DYNAMIC_STEP,300.0,1.0,301.0,301.0
255,DY-STEP-004,DYNAMIC_STEP,300.0,1.0,301.0,301.0
256,DY-STEP-005,DYNAMIC_STEP,300.0,1.0,301.0,301.0
257,DY-STEP-006,DYNAMIC_STEP,300.0,1.0,301.0,301.0
258,DY-STEP-007,DYNAMIC_STEP,300.0,1.0,301.0,301.0
259,DY-STEP-008,DYNAMIC_STEP,300.0,1.0,301.0,301.0
260,DY-STEP-009,DYNAMIC_STEP,300.0,1.0,301.0,301.0
261,DY-STEP-010,DYNAMIC_STEP,300.0,1.0,301.0,301.0


In [131]:
count_check_df = test_completeness_df[
    test_completeness_df[
        "Derived_Expected_Samples"
    ].notna()
].copy()

count_check_df["Sample_Difference"] = (
    count_check_df["Observed_Samples"]
    - count_check_df["Derived_Expected_Samples"]
)

count_check_df["Completeness_Status"] = (
    count_check_df["Sample_Difference"]
    .eq(0)
    .map({
        True: "PASS",
        False: "FAIL"
    })
)

In [132]:
completeness_summary = (
    count_check_df
    .groupby("Test_Family")
    .agg(
        Tests_Checked=("Test_ID", "size"),
        Tests_Passed=(
            "Completeness_Status",
            lambda x: (x == "PASS").sum()
        ),
        Tests_Failed=(
            "Completeness_Status",
            lambda x: (x == "FAIL").sum()
        ),
    )
    .reset_index()
)

display(completeness_summary)

explicit_count_pass = (
    count_check_df["Completeness_Status"] == "PASS"
).all()

print(
    f"Explicit/derived sample-count qualification : "
    f"{'PASS' if explicit_count_pass else 'FAIL'}"
)

,Test_Family,Tests_Checked,Tests_Passed,Tests_Failed
0,DYNAMIC_RAMP,18,18,0
1,DYNAMIC_STEP,30,30,0
2,STATIC,252,252,0


Explicit/derived sample-count qualification : PASS


### 4.5 Completeness Status for Remaining Test Families

Strict sample-count completeness was verified for all test families for which
the experimental design provides either an explicit or defensibly derived
expected sample count.

Results:

- `STATIC`: strict sample-count completeness verified
- `DYNAMIC_STEP`: strict sample-count completeness verified
- `DYNAMIC_RAMP`: strict sample-count completeness verified

The remaining measurement families require different qualification criteria:

- `DYNAMIC_CYCLIC`: completeness will be evaluated using cycle coverage and
  sequence integrity.
- `DYNAMIC_SINUSOIDAL`: completeness will be evaluated using temporal
  continuity and consistency with the commanded period/frequency.
- `STABILITY`: completeness will be evaluated against the scheduled aging-day
  and stability-test structure.

`UNCERTAINTY_EVALUATION` entries are design points and are not expected to
produce standalone measurement records.

Therefore, absence of `Expected_Samples` for these families is not treated as
a data-quality failure.

In [133]:
# ------------------------------------------------------------
# Record family-specific completeness disposition
# ------------------------------------------------------------

family_completeness_status = pd.DataFrame(
    [
        {
            "Test_Family": "STATIC",
            "Qualification_Method": "Explicit expected sample count",
            "Status": "PASS",
        },
        {
            "Test_Family": "DYNAMIC_STEP",
            "Qualification_Method": "Duration x sample rate + initial sample",
            "Status": "PASS",
        },
        {
            "Test_Family": "DYNAMIC_RAMP",
            "Qualification_Method": "Duration x sample rate + initial sample",
            "Status": "PASS",
        },
        {
            "Test_Family": "DYNAMIC_CYCLIC",
            "Qualification_Method": "Cycle coverage / sequence integrity",
            "Status": "DEFERRED",
        },
        {
            "Test_Family": "DYNAMIC_SINUSOIDAL",
            "Qualification_Method": "Temporal / period-frequency integrity",
            "Status": "DEFERRED",
        },
        {
            "Test_Family": "STABILITY",
            "Qualification_Method": "Schedule / aging-day integrity",
            "Status": "DEFERRED",
        },
        {
            "Test_Family": "UNCERTAINTY_EVALUATION",
            "Qualification_Method": "Design-only points",
            "Status": "NOT_APPLICABLE",
        },
    ]
)

display(family_completeness_status)

,Test_Family,Qualification_Method,Status
0,STATIC,Explicit expected sample count,PASS
1,DYNAMIC_STEP,Duration x sample rate + initial sample,PASS
2,DYNAMIC_RAMP,Duration x sample rate + initial sample,PASS
3,DYNAMIC_CYCLIC,Cycle coverage / sequence integrity,DEFERRED
4,DYNAMIC_SINUSOIDAL,Temporal / period-frequency integrity,DEFERRED
5,STABILITY,Schedule / aging-day integrity,DEFERRED
6,UNCERTAINTY_EVALUATION,Design-only points,NOT_APPLICABLE


## 5. Calibration / Validation Role Separation

### Engineering Question

Are calibration, validation, and characterization observations correctly
identified and kept methodologically separate?

### Why This Check Matters

Calibration data are used to develop the correction model, whereas validation
data must remain independent until the calibration model has been finalized.

If the same observations are used for both model development and validation,
the apparent validation performance may be optimistically biased and cannot
be considered an independent assessment.

This check verifies that the analysis role recorded in the static measurement
data agrees with the role defined in the experimental test matrix.

In [134]:
# ------------------------------------------------------------
# Inspect analysis roles in static measurements and test matrix
# ------------------------------------------------------------

static_df = datasets["static_measurements.csv"]

static_roles = (
    static_df["Analysis_Role"]
    .value_counts()
    .rename_axis("Analysis_Role")
    .reset_index(name="Measurement_Rows")
)

matrix_static_roles = (
    test_matrix_df[
        test_matrix_df["Test_Family"] == "STATIC"
    ]["Analysis_Role"]
    .value_counts()
    .rename_axis("Analysis_Role")
    .reset_index(name="Tests")
)

print("Static measurement roles:")
display(static_roles)

print("Static test-matrix roles:")
display(matrix_static_roles)

Static measurement roles:


,Analysis_Role,Measurement_Rows
0,CHARACTERIZATION,7200
1,CALIBRATION,5760
2,VALIDATION,2160


Static test-matrix roles:


,Analysis_Role,Tests
0,CHARACTERIZATION,120
1,CALIBRATION,96
2,VALIDATION,36


In [135]:
# ------------------------------------------------------------
# Verify role consistency for every static Test_ID
# ------------------------------------------------------------

measurement_role_by_test = (
    static_df[
        ["Test_ID", "Analysis_Role"]
    ]
    .drop_duplicates()
)

matrix_role_by_test = (
    test_matrix_df[
        test_matrix_df["Test_Family"] == "STATIC"
    ][
        ["Test_ID", "Analysis_Role"]
    ]
)

role_check_df = measurement_role_by_test.merge(
    matrix_role_by_test,
    on="Test_ID",
    how="outer",
    suffixes=("_Measurement", "_Matrix"),
    indicator=True
)

role_check_df["Role_Matches"] = (
    role_check_df["Analysis_Role_Measurement"]
    == role_check_df["Analysis_Role_Matrix"]
)

display(role_check_df.head(10))

,Test_ID,Analysis_Role_Measurement,Analysis_Role_Matrix,_merge,Role_Matches
0,CAL-001,CALIBRATION,CALIBRATION,both,True
1,CAL-002,CALIBRATION,CALIBRATION,both,True
2,CAL-003,CALIBRATION,CALIBRATION,both,True
3,CAL-004,CALIBRATION,CALIBRATION,both,True
4,CAL-005,CALIBRATION,CALIBRATION,both,True
5,CAL-006,CALIBRATION,CALIBRATION,both,True
6,CAL-007,CALIBRATION,CALIBRATION,both,True
7,CAL-008,CALIBRATION,CALIBRATION,both,True
8,CAL-009,CALIBRATION,CALIBRATION,both,True
9,CAL-010,CALIBRATION,CALIBRATION,both,True


In [136]:
# ------------------------------------------------------------
# Summarize role-mapping integrity
# ------------------------------------------------------------

unmatched_role_tests = role_check_df[
    role_check_df["_merge"] != "both"
]

role_mismatches = role_check_df[
    (role_check_df["_merge"] == "both")
    & (~role_check_df["Role_Matches"])
]

role_mapping_pass = (
    len(unmatched_role_tests) == 0
    and len(role_mismatches) == 0
)

print(
    f"Static Test_IDs checked       : "
    f"{len(role_check_df)}"
)

print(
    f"Unmatched static Test_IDs     : "
    f"{len(unmatched_role_tests)}"
)

print(
    f"Role mismatches               : "
    f"{len(role_mismatches)}"
)

print(
    f"\nAnalysis-role mapping         : "
    f"{'PASS' if role_mapping_pass else 'FAIL'}"
)

Static Test_IDs checked       : 252
Unmatched static Test_IDs     : 0
Role mismatches               : 0

Analysis-role mapping         : PASS


In [137]:
# ------------------------------------------------------------
# Verify that each static Test_ID has exactly one analysis role
# ------------------------------------------------------------

roles_per_test = (
    static_df
    .groupby("Test_ID")["Analysis_Role"]
    .nunique()
)

multi_role_test_ids = roles_per_test[
    roles_per_test > 1
]

single_role_pass = (
    len(multi_role_test_ids) == 0
)

print(
    f"Test_IDs assigned to multiple roles : "
    f"{len(multi_role_test_ids)}"
)

print(
    f"Single-role integrity               : "
    f"{'PASS' if single_role_pass else 'FAIL'}"
)

Test_IDs assigned to multiple roles : 0
Single-role integrity               : PASS


### 5.1 Role Isolation

Correct role labels are necessary but not sufficient for independent
validation.

The sets of `Test_ID` values assigned to calibration, validation, and
characterization must also be mutually exclusive. In particular, no
validation test may also appear in the calibration set.

This check verifies experimental partition isolation before any calibration
model is developed.

In [138]:
# ------------------------------------------------------------
# Verify that analysis-role Test_ID sets are mutually exclusive
# ------------------------------------------------------------

calibration_ids = set(
    static_df.loc[
        static_df["Analysis_Role"] == "CALIBRATION",
        "Test_ID"
    ].unique()
)

validation_ids = set(
    static_df.loc[
        static_df["Analysis_Role"] == "VALIDATION",
        "Test_ID"
    ].unique()
)

characterization_ids = set(
    static_df.loc[
        static_df["Analysis_Role"] == "CHARACTERIZATION",
        "Test_ID"
    ].unique()
)

cal_val_overlap = calibration_ids & validation_ids
cal_char_overlap = calibration_ids & characterization_ids
val_char_overlap = validation_ids & characterization_ids

print(f"Calibration Test_IDs      : {len(calibration_ids)}")
print(f"Validation Test_IDs       : {len(validation_ids)}")
print(f"Characterization Test_IDs : {len(characterization_ids)}")

print(f"\nCalibration / Validation overlap      : {len(cal_val_overlap)}")
print(f"Calibration / Characterization overlap: {len(cal_char_overlap)}")
print(f"Validation / Characterization overlap : {len(val_char_overlap)}")

Calibration Test_IDs      : 96
Validation Test_IDs       : 36
Characterization Test_IDs : 120

Calibration / Validation overlap      : 0
Calibration / Characterization overlap: 0
Validation / Characterization overlap : 0


In [139]:
role_isolation_pass = (
    len(cal_val_overlap) == 0
    and len(cal_char_overlap) == 0
    and len(val_char_overlap) == 0
)

print(
    f"\nAnalysis-role isolation : "
    f"{'PASS' if role_isolation_pass else 'FAIL'}"
)


Analysis-role isolation : PASS


## 6. Time & Sequence Integrity

### Engineering Question

Are measurement records temporally ordered and internally consistent within
each experimental test?

### Why This Check Matters

Dynamic-response, tracking, cyclic, and stability analyses depend on the
correct temporal ordering of measurement records.

Before estimating response time, phase lag, drift, or other time-dependent
quantities, each test must have a coherent elapsed-time sequence.

This section evaluates temporal integrity only. Sensor dynamic performance is
not assessed here.

In [140]:
# ------------------------------------------------------------
# Audit elapsed-time sequence integrity within each Test_ID
# ------------------------------------------------------------

time_integrity_results = []

for file_name in measurement_files:

    df = datasets[file_name]

    for test_id, group in df.groupby("Test_ID"):

        group = group.sort_values("Sample_ID")

        elapsed = group["Elapsed_s"]
        time_differences = elapsed.diff().dropna()

        starts_at_zero = (
            len(elapsed) > 0
            and elapsed.iloc[0] == 0
        )

        strictly_increasing = (
            (time_differences > 0).all()
            if len(time_differences) > 0
            else True
        )

        duplicate_elapsed_values = int(
            elapsed.duplicated().sum()
        )

        median_sample_interval = (
            time_differences.median()
            if len(time_differences) > 0
            else float("nan")
        )

        time_integrity_results.append(
            {
                "Source_File": file_name,
                "Test_ID": test_id,
                "Starts_At_Zero": starts_at_zero,
                "Strictly_Increasing": strictly_increasing,
                "Duplicate_Elapsed_Values": duplicate_elapsed_values,
                "Median_dt_s": median_sample_interval,
            }
        )

time_integrity_df = pd.DataFrame(
    time_integrity_results
)

display(time_integrity_df.head(10))

,Source_File,Test_ID,Starts_At_Zero,Strictly_Increasing,Duplicate_Elapsed_Values,Median_dt_s
0,cyclic_measurements.csv,DY-CYC-001,True,True,0,1.0
1,cyclic_measurements.csv,DY-CYC-002,True,True,0,1.0
2,cyclic_measurements.csv,DY-CYC-003,True,True,0,1.0
3,dynamic_step_measurements.csv,DY-STEP-001,True,True,0,1.0
4,dynamic_step_measurements.csv,DY-STEP-002,True,True,0,1.0
5,dynamic_step_measurements.csv,DY-STEP-003,True,True,0,1.0
6,dynamic_step_measurements.csv,DY-STEP-004,True,True,0,1.0
7,dynamic_step_measurements.csv,DY-STEP-005,True,True,0,1.0
8,dynamic_step_measurements.csv,DY-STEP-006,True,True,0,1.0
9,dynamic_step_measurements.csv,DY-STEP-007,True,True,0,1.0


In [141]:
# ------------------------------------------------------------
# Summarize temporal-sequence integrity
# ------------------------------------------------------------

tests_not_starting_zero = (
    ~time_integrity_df["Starts_At_Zero"]
).sum()

tests_not_monotonic = (
    ~time_integrity_df["Strictly_Increasing"]
).sum()

duplicate_elapsed_values = (
    time_integrity_df[
        "Duplicate_Elapsed_Values"
    ].sum()
)

time_sequence_pass = (
    tests_not_starting_zero == 0
    and tests_not_monotonic == 0
    and duplicate_elapsed_values == 0
)

print(
    f"Tests checked                    : "
    f"{len(time_integrity_df)}"
)

print(
    f"Tests not starting at t = 0      : "
    f"{tests_not_starting_zero}"
)

print(
    f"Tests with non-monotonic time    : "
    f"{tests_not_monotonic}"
)

print(
    f"Duplicate elapsed-time values    : "
    f"{duplicate_elapsed_values}"
)

print(
    f"\nBasic time-sequence integrity     : "
    f"{'PASS' if time_sequence_pass else 'FAIL'}"
)

Tests checked                    : 329
Tests not starting at t = 0      : 0
Tests with non-monotonic time    : 0
Duplicate elapsed-time values    : 0

Basic time-sequence integrity     : PASS


### 6.1 Sampling-Interval Consistency

A monotonic time sequence does not by itself guarantee complete temporal
sampling. Missing or irregular intervals may still exist within an otherwise
ordered test.

For tests with a defined `Sample_Rate_Hz` in the experimental design, the
observed elapsed-time increments are compared with the expected sampling
interval:

$$
\Delta t_{\mathrm{expected}} =
\frac{1}{f_s}
$$

This check verifies temporal sampling integrity without evaluating sensor
dynamic performance.

In [142]:
# ------------------------------------------------------------
# Compare observed sampling intervals with test-matrix design
# ------------------------------------------------------------

sampling_interval_results = []

for file_name in measurement_files:

    df = datasets[file_name]

    for test_id, group in df.groupby("Test_ID"):

        design_row = test_matrix_df[
            test_matrix_df["Test_ID"] == test_id
        ]

        if design_row.empty:
            continue

        sample_rate = design_row["Sample_Rate_Hz"].iloc[0]

        # Only perform strict interval checking when the
        # experimental design defines a sample rate.
        if pd.isna(sample_rate):
            continue

        expected_dt = 1.0 / sample_rate

        group = group.sort_values("Sample_ID")

        observed_dt = (
            group["Elapsed_s"]
            .diff()
            .dropna()
        )

        interval_mismatches = int(
            (~observed_dt.eq(expected_dt)).sum()
        )

        sampling_interval_results.append(
            {
                "Test_ID": test_id,
                "Test_Family": group["Test_Family"].iloc[0],
                "Expected_dt_s": expected_dt,
                "Observed_Min_dt_s": observed_dt.min(),
                "Observed_Median_dt_s": observed_dt.median(),
                "Observed_Max_dt_s": observed_dt.max(),
                "Interval_Mismatches": interval_mismatches,
                "Sampling_Status": (
                    "PASS"
                    if interval_mismatches == 0
                    else "FAIL"
                ),
            }
        )

sampling_interval_df = pd.DataFrame(
    sampling_interval_results
)

display(sampling_interval_df.head(10))

,Test_ID,Test_Family,Expected_dt_s,Observed_Min_dt_s,Observed_Median_dt_s,Observed_Max_dt_s,Interval_Mismatches,Sampling_Status
0,DY-CYC-001,DYNAMIC_CYCLIC,1.0,1.0,1.0,1.0,0,PASS
1,DY-CYC-002,DYNAMIC_CYCLIC,1.0,1.0,1.0,1.0,0,PASS
2,DY-CYC-003,DYNAMIC_CYCLIC,1.0,1.0,1.0,1.0,0,PASS
3,DY-STEP-001,DYNAMIC_STEP,1.0,1.0,1.0,1.0,0,PASS
4,DY-STEP-002,DYNAMIC_STEP,1.0,1.0,1.0,1.0,0,PASS
5,DY-STEP-003,DYNAMIC_STEP,1.0,1.0,1.0,1.0,0,PASS
6,DY-STEP-004,DYNAMIC_STEP,1.0,1.0,1.0,1.0,0,PASS
7,DY-STEP-005,DYNAMIC_STEP,1.0,1.0,1.0,1.0,0,PASS
8,DY-STEP-006,DYNAMIC_STEP,1.0,1.0,1.0,1.0,0,PASS
9,DY-STEP-007,DYNAMIC_STEP,1.0,1.0,1.0,1.0,0,PASS


In [143]:
sampling_summary = (
    sampling_interval_df
    .groupby("Test_Family")
    .agg(
        Tests_Checked=("Test_ID", "size"),
        Tests_Passed=(
            "Sampling_Status",
            lambda x: (x == "PASS").sum()
        ),
        Tests_Failed=(
            "Sampling_Status",
            lambda x: (x == "FAIL").sum()
        ),
        Total_Interval_Mismatches=(
            "Interval_Mismatches",
            "sum"
        ),
    )
    .reset_index()
)

display(sampling_summary)

sampling_interval_pass = (
    sampling_interval_df["Sampling_Status"] == "PASS"
).all()

print(
    f"Sampling-interval qualification : "
    f"{'PASS' if sampling_interval_pass else 'FAIL'}"
)

,Test_Family,Tests_Checked,Tests_Passed,Tests_Failed,Total_Interval_Mismatches
0,DYNAMIC_CYCLIC,3,3,0,0
1,DYNAMIC_RAMP,18,18,0,0
2,DYNAMIC_SINUSOIDAL,4,4,0,0
3,DYNAMIC_STEP,30,30,0,0
4,STATIC,252,252,0,0


Sampling-interval qualification : PASS


### 6.2 Cyclic-Test Coverage

The cyclic test design specifies the required number of humidity cycles for
each cyclic `Test_ID`.

Because `cyclic_measurements.csv` contains a `Cycle_Index`, cyclic completeness
can be evaluated by comparing the observed cycle coverage with the
`Cycle_Count` defined in the experimental test matrix.

This check evaluates execution completeness only. Dynamic hysteresis,
cycle-to-cycle repeatability, and sensor tracking performance are outside the
scope of DQ-001.

In [144]:
# ------------------------------------------------------------
# Verify cyclic-test cycle coverage against experimental design
# ------------------------------------------------------------

cyclic_df = datasets["cyclic_measurements.csv"]

cyclic_design = test_matrix_df[
    test_matrix_df["Test_Family"] == "DYNAMIC_CYCLIC"
][
    ["Test_ID", "Session", "Cycle_Count"]
].copy()

cyclic_observed = (
    cyclic_df
    .groupby("Test_ID")
    .agg(
        Observed_Cycles=("Cycle_Index", "nunique"),
        Min_Cycle_Index=("Cycle_Index", "min"),
        Max_Cycle_Index=("Cycle_Index", "max"),
        Observed_Sessions=("Session", "nunique"),
        Session_Value=("Session", "first"),
    )
    .reset_index()
)

cyclic_check_df = cyclic_design.merge(
    cyclic_observed,
    on="Test_ID",
    how="outer",
    indicator=True
)

cyclic_check_df["Cycle_Count_Matches"] = (
    cyclic_check_df["Observed_Cycles"]
    == cyclic_check_df["Cycle_Count"]
)

cyclic_check_df["Session_Matches"] = (
    cyclic_check_df["Session"]
    == cyclic_check_df["Session_Value"]
)

display(cyclic_check_df)

,Test_ID,Session,Cycle_Count,Observed_Cycles,Min_Cycle_Index,Max_Cycle_Index,Observed_Sessions,Session_Value,_merge,Cycle_Count_Matches,Session_Matches
0,DY-CYC-001,1.0,5.0,5,1,5,1,1,both,True,True
1,DY-CYC-002,2.0,5.0,5,1,5,1,2,both,True,True
2,DY-CYC-003,3.0,5.0,5,1,5,1,3,both,True,True


In [145]:
cyclic_coverage_pass = (
    (cyclic_check_df["_merge"] == "both").all()
    and cyclic_check_df["Cycle_Count_Matches"].all()
    and cyclic_check_df["Session_Matches"].all()
)

print(
    f"Cyclic tests checked        : "
    f"{len(cyclic_check_df)}"
)

print(
    f"Cycle-count mismatches      : "
    f"{(~cyclic_check_df['Cycle_Count_Matches']).sum()}"
)

print(
    f"Session mismatches          : "
    f"{(~cyclic_check_df['Session_Matches']).sum()}"
)

print(
    f"\nCyclic coverage qualification : "
    f"{'PASS' if cyclic_coverage_pass else 'FAIL'}"
)

Cyclic tests checked        : 3
Cycle-count mismatches      : 0
Session mismatches          : 0

Cyclic coverage qualification : PASS


### 6.3 Sinusoidal-Test Integrity

Sinusoidal dynamic tests are defined by their commanded period and frequency.

For each sinusoidal `Test_ID`, this check verifies that:

- the recorded period is internally constant within the test
- the recorded frequency is internally constant within the test
- the recorded period agrees with the experimental test matrix
- the recorded frequency agrees with the experimental test matrix
- the period-frequency relationship is physically consistent with

\[
f = \frac{1}{T}
\]

This is a qualification of experimental metadata and temporal structure.
Sensor gain and phase lag are not evaluated in DQ-001.

In [146]:
# ------------------------------------------------------------
# Verify sinusoidal period/frequency consistency
# ------------------------------------------------------------

sinusoidal_df = datasets["sinusoidal_measurements.csv"]

sinusoidal_design = test_matrix_df[
    test_matrix_df["Test_Family"] == "DYNAMIC_SINUSOIDAL"
][
    ["Test_ID", "Period_s", "Frequency_Hz"]
].copy()

sinusoidal_observed = (
    sinusoidal_df
    .groupby("Test_ID")
    .agg(
        Observed_Period_s=("Period_s", "first"),
        Unique_Period_Values=("Period_s", "nunique"),
        Observed_Frequency_Hz=("Frequency_Hz", "first"),
        Unique_Frequency_Values=("Frequency_Hz", "nunique"),
        Start_Time_s=("Elapsed_s", "min"),
        End_Time_s=("Elapsed_s", "max"),
        Observed_Samples=("Sample_ID", "size"),
    )
    .reset_index()
)

sinusoidal_check_df = sinusoidal_design.merge(
    sinusoidal_observed,
    on="Test_ID",
    how="outer",
    indicator=True
)

sinusoidal_check_df["Period_Internal_Consistency"] = (
    sinusoidal_check_df["Unique_Period_Values"] == 1
)

sinusoidal_check_df["Frequency_Internal_Consistency"] = (
    sinusoidal_check_df["Unique_Frequency_Values"] == 1
)

sinusoidal_check_df["Period_Matches_Design"] = (
    sinusoidal_check_df["Observed_Period_s"]
    == sinusoidal_check_df["Period_s"]
)

sinusoidal_check_df["Frequency_Matches_Design"] = (
    (
        sinusoidal_check_df["Observed_Frequency_Hz"]
        - sinusoidal_check_df["Frequency_Hz"]
    ).abs()
    <= 1e-6
)

display(sinusoidal_check_df)

,Test_ID,Period_s,Frequency_Hz,Observed_Period_s,Unique_Period_Values,Observed_Frequency_Hz,Unique_Frequency_Values,Start_Time_s,End_Time_s,Observed_Samples,_merge,Period_Internal_Consistency,Frequency_Internal_Consistency,Period_Matches_Design,Frequency_Matches_Design
0,DY-SIN-001,600.0,0.001667,600,1,0.001667,1,0.0,3000.0,3001,both,True,True,True,True
1,DY-SIN-002,300.0,0.003333,300,1,0.003333,1,0.0,1500.0,1501,both,True,True,True,True
2,DY-SIN-003,120.0,0.008333,120,1,0.008333,1,0.0,600.0,601,both,True,True,True,True
3,DY-SIN-004,60.0,0.016667,60,1,0.016667,1,0.0,300.0,301,both,True,True,True,True


In [147]:
# ------------------------------------------------------------
# Check physical consistency between period and frequency
# ------------------------------------------------------------

sinusoidal_check_df["Frequency_From_Period_Hz"] = (
    1.0 / sinusoidal_check_df["Observed_Period_s"]
)

sinusoidal_check_df["Period_Frequency_Error_Hz"] = (
    sinusoidal_check_df["Observed_Frequency_Hz"]
    - sinusoidal_check_df["Frequency_From_Period_Hz"]
).abs()

sinusoidal_check_df["Period_Frequency_Consistent"] = (
    sinusoidal_check_df["Period_Frequency_Error_Hz"]
    <= 1e-6
)

display(
    sinusoidal_check_df[
        [
            "Test_ID",
            "Observed_Period_s",
            "Observed_Frequency_Hz",
            "Frequency_From_Period_Hz",
            "Period_Frequency_Error_Hz",
            "Period_Frequency_Consistent",
        ]
    ]
)

,Test_ID,Observed_Period_s,Observed_Frequency_Hz,Frequency_From_Period_Hz,Period_Frequency_Error_Hz,Period_Frequency_Consistent
0,DY-SIN-001,600,0.001667,0.001667,3.333333e-07,True
1,DY-SIN-002,300,0.003333,0.003333,3.333333e-07,True
2,DY-SIN-003,120,0.008333,0.008333,3.333333e-07,True
3,DY-SIN-004,60,0.016667,0.016667,3.333333e-07,True


In [148]:
# ------------------------------------------------------------
# Summarize sinusoidal-test integrity
# ------------------------------------------------------------

sinusoidal_integrity_pass = (
    (sinusoidal_check_df["_merge"] == "both").all()
    and sinusoidal_check_df[
        "Period_Internal_Consistency"
    ].all()
    and sinusoidal_check_df[
        "Frequency_Internal_Consistency"
    ].all()
    and sinusoidal_check_df[
        "Period_Matches_Design"
    ].all()
    and sinusoidal_check_df[
        "Frequency_Matches_Design"
    ].all()
    and sinusoidal_check_df[
        "Period_Frequency_Consistent"
    ].all()
)

print(
    f"Sinusoidal tests checked              : "
    f"{len(sinusoidal_check_df)}"
)

print(
    f"Tests with varying period             : "
    f"{(~sinusoidal_check_df['Period_Internal_Consistency']).sum()}"
)

print(
    f"Tests with varying frequency          : "
    f"{(~sinusoidal_check_df['Frequency_Internal_Consistency']).sum()}"
)

print(
    f"Period/design mismatches              : "
    f"{(~sinusoidal_check_df['Period_Matches_Design']).sum()}"
)

print(
    f"Frequency/design mismatches           : "
    f"{(~sinusoidal_check_df['Frequency_Matches_Design']).sum()}"
)

print(
    f"Period-frequency inconsistencies      : "
    f"{(~sinusoidal_check_df['Period_Frequency_Consistent']).sum()}"
)

print(
    f"\nSinusoidal integrity qualification    : "
    f"{'PASS' if sinusoidal_integrity_pass else 'FAIL'}"
)

Sinusoidal tests checked              : 4
Tests with varying period             : 0
Tests with varying frequency          : 0
Period/design mismatches              : 0
Frequency/design mismatches           : 0
Period-frequency inconsistencies      : 0

Sinusoidal integrity qualification    : PASS


### 6.4 Stability Schedule Integrity

Stability measurements are collected according to a predefined aging schedule.

Before estimating sensor drift, each stability test must be traceable to the
intended aging day, schedule type, humidity setpoint, and temperature setpoint.

This check verifies that:

- every planned stability `Test_ID` has measurement records
- schedule metadata are internally constant within each test
- recorded schedule metadata agree with the experimental test matrix
- weekly and monthly stability-test coverage is preserved

Sensor drift magnitude is not evaluated in DQ-001.

In [149]:
# ------------------------------------------------------------
# Build stability design and observed metadata summaries
# ------------------------------------------------------------

stability_df = datasets["stability_measurements.csv"]

stability_design = test_matrix_df[
    test_matrix_df["Test_Family"] == "STABILITY"
][
    [
        "Test_ID",
        "Schedule_Type",
        "Aging_Day",
        "RH_Setpoint",
        "Temperature_Setpoint_C",
    ]
].copy()

stability_observed = (
    stability_df
    .groupby("Test_ID")
    .agg(
        Observed_Schedule_Type=("Schedule_Type", "first"),
        Unique_Schedule_Types=("Schedule_Type", "nunique"),
        Observed_Aging_Day=("Aging_Day", "first"),
        Unique_Aging_Days=("Aging_Day", "nunique"),
        Observed_RH_Setpoint=("RH_Setpoint", "first"),
        Unique_RH_Setpoints=("RH_Setpoint", "nunique"),
        Observed_Temperature_Setpoint_C=(
            "Temperature_Setpoint_C",
            "first"
        ),
        Unique_Temperature_Setpoints=(
            "Temperature_Setpoint_C",
            "nunique"
        ),
        Observed_Samples=("Sample_ID", "size"),
    )
    .reset_index()
)

stability_check_df = stability_design.merge(
    stability_observed,
    on="Test_ID",
    how="outer",
    indicator=True
)

display(stability_check_df)

,Test_ID,Schedule_Type,Aging_Day,RH_Setpoint,Temperature_Setpoint_C,Observed_Schedule_Type,Unique_Schedule_Types,Observed_Aging_Day,Unique_Aging_Days,Observed_RH_Setpoint,Unique_RH_Setpoints,Observed_Temperature_Setpoint_C,Unique_Temperature_Setpoints,Observed_Samples,_merge
0,STAB-001,WEEKLY_CENTER_CHECK,0.0,50.0,25.0,WEEKLY_CENTER_CHECK,1,0,1,50.0,1,25.0,1,60,both
1,STAB-002,WEEKLY_CENTER_CHECK,7.0,50.0,25.0,WEEKLY_CENTER_CHECK,1,7,1,50.0,1,25.0,1,60,both
2,STAB-003,WEEKLY_CENTER_CHECK,14.0,50.0,25.0,WEEKLY_CENTER_CHECK,1,14,1,50.0,1,25.0,1,60,both
3,STAB-004,WEEKLY_CENTER_CHECK,21.0,50.0,25.0,WEEKLY_CENTER_CHECK,1,21,1,50.0,1,25.0,1,60,both
4,STAB-005,WEEKLY_CENTER_CHECK,28.0,50.0,25.0,WEEKLY_CENTER_CHECK,1,28,1,50.0,1,25.0,1,60,both
5,STAB-006,WEEKLY_CENTER_CHECK,35.0,50.0,25.0,WEEKLY_CENTER_CHECK,1,35,1,50.0,1,25.0,1,60,both
6,STAB-007,WEEKLY_CENTER_CHECK,42.0,50.0,25.0,WEEKLY_CENTER_CHECK,1,42,1,50.0,1,25.0,1,60,both
7,STAB-008,WEEKLY_CENTER_CHECK,49.0,50.0,25.0,WEEKLY_CENTER_CHECK,1,49,1,50.0,1,25.0,1,60,both
8,STAB-009,WEEKLY_CENTER_CHECK,56.0,50.0,25.0,WEEKLY_CENTER_CHECK,1,56,1,50.0,1,25.0,1,60,both
9,STAB-010,WEEKLY_CENTER_CHECK,63.0,50.0,25.0,WEEKLY_CENTER_CHECK,1,63,1,50.0,1,25.0,1,60,both


In [150]:
# ------------------------------------------------------------
# Evaluate stability metadata integrity
# ------------------------------------------------------------

stability_check_df["Schedule_Internal_Consistency"] = (
    stability_check_df["Unique_Schedule_Types"] == 1
)

stability_check_df["Aging_Day_Internal_Consistency"] = (
    stability_check_df["Unique_Aging_Days"] == 1
)

stability_check_df["RH_Internal_Consistency"] = (
    stability_check_df["Unique_RH_Setpoints"] == 1
)

stability_check_df["Temperature_Internal_Consistency"] = (
    stability_check_df["Unique_Temperature_Setpoints"] == 1
)

stability_check_df["Schedule_Matches_Design"] = (
    stability_check_df["Schedule_Type"]
    == stability_check_df["Observed_Schedule_Type"]
)

stability_check_df["Aging_Day_Matches_Design"] = (
    stability_check_df["Aging_Day"]
    == stability_check_df["Observed_Aging_Day"]
)

stability_check_df["RH_Setpoint_Matches_Design"] = (
    stability_check_df["RH_Setpoint"]
    == stability_check_df["Observed_RH_Setpoint"]
)

stability_check_df["Temperature_Matches_Design"] = (
    stability_check_df["Temperature_Setpoint_C"]
    == stability_check_df["Observed_Temperature_Setpoint_C"]
)

display(
    stability_check_df[
        [
            "Test_ID",
            "Schedule_Type",
            "Aging_Day",
            "RH_Setpoint",
            "Observed_Samples",
            "Schedule_Matches_Design",
            "Aging_Day_Matches_Design",
            "RH_Setpoint_Matches_Design",
            "Temperature_Matches_Design",
        ]
    ]
)

,Test_ID,Schedule_Type,Aging_Day,RH_Setpoint,Observed_Samples,Schedule_Matches_Design,Aging_Day_Matches_Design,RH_Setpoint_Matches_Design,Temperature_Matches_Design
0,STAB-001,WEEKLY_CENTER_CHECK,0.0,50.0,60,True,True,True,True
1,STAB-002,WEEKLY_CENTER_CHECK,7.0,50.0,60,True,True,True,True
2,STAB-003,WEEKLY_CENTER_CHECK,14.0,50.0,60,True,True,True,True
3,STAB-004,WEEKLY_CENTER_CHECK,21.0,50.0,60,True,True,True,True
4,STAB-005,WEEKLY_CENTER_CHECK,28.0,50.0,60,True,True,True,True
5,STAB-006,WEEKLY_CENTER_CHECK,35.0,50.0,60,True,True,True,True
6,STAB-007,WEEKLY_CENTER_CHECK,42.0,50.0,60,True,True,True,True
7,STAB-008,WEEKLY_CENTER_CHECK,49.0,50.0,60,True,True,True,True
8,STAB-009,WEEKLY_CENTER_CHECK,56.0,50.0,60,True,True,True,True
9,STAB-010,WEEKLY_CENTER_CHECK,63.0,50.0,60,True,True,True,True


In [151]:
# ------------------------------------------------------------
# Summarize stability schedule integrity
# ------------------------------------------------------------

stability_integrity_pass = (
    (stability_check_df["_merge"] == "both").all()
    and stability_check_df[
        "Schedule_Internal_Consistency"
    ].all()
    and stability_check_df[
        "Aging_Day_Internal_Consistency"
    ].all()
    and stability_check_df[
        "RH_Internal_Consistency"
    ].all()
    and stability_check_df[
        "Temperature_Internal_Consistency"
    ].all()
    and stability_check_df[
        "Schedule_Matches_Design"
    ].all()
    and stability_check_df[
        "Aging_Day_Matches_Design"
    ].all()
    and stability_check_df[
        "RH_Setpoint_Matches_Design"
    ].all()
    and stability_check_df[
        "Temperature_Matches_Design"
    ].all()
)

print(
    f"Stability tests checked               : "
    f"{len(stability_check_df)}"
)

print(
    f"Missing/unmatched stability tests     : "
    f"{(stability_check_df['_merge'] != 'both').sum()}"
)

print(
    f"Schedule mismatches                   : "
    f"{(~stability_check_df['Schedule_Matches_Design']).sum()}"
)

print(
    f"Aging-day mismatches                  : "
    f"{(~stability_check_df['Aging_Day_Matches_Design']).sum()}"
)

print(
    f"RH-setpoint mismatches                : "
    f"{(~stability_check_df['RH_Setpoint_Matches_Design']).sum()}"
)

print(
    f"Temperature-setpoint mismatches       : "
    f"{(~stability_check_df['Temperature_Matches_Design']).sum()}"
)

print(
    f"\nStability schedule qualification      : "
    f"{'PASS' if stability_integrity_pass else 'FAIL'}"
)

Stability tests checked               : 22
Missing/unmatched stability tests     : 0
Schedule mismatches                   : 0
Aging-day mismatches                  : 0
RH-setpoint mismatches                : 0
Temperature-setpoint mismatches       : 0

Stability schedule qualification      : PASS


In [152]:
# ------------------------------------------------------------
# Inspect stability aging-schedule coverage
# ------------------------------------------------------------

stability_schedule_coverage = (
    stability_check_df
    .groupby("Schedule_Type")
    .agg(
        Tests=("Test_ID", "size"),
        Min_Aging_Day=("Aging_Day", "min"),
        Max_Aging_Day=("Aging_Day", "max"),
        Unique_Aging_Days=("Aging_Day", "nunique"),
        Unique_RH_Setpoints=("RH_Setpoint", "nunique"),
    )
    .reset_index()
)

display(stability_schedule_coverage)

,Schedule_Type,Tests,Min_Aging_Day,Max_Aging_Day,Unique_Aging_Days,Unique_RH_Setpoints
0,MONTHLY_MULTI_POINT,9,30.0,90.0,3,3
1,WEEKLY_CENTER_CHECK,13,0.0,84.0,13,1


In [153]:
# ------------------------------------------------------------
# Finalize family-specific completeness disposition
# ------------------------------------------------------------

family_completeness_final = pd.DataFrame(
    [
        {
            "Test_Family": "STATIC",
            "Qualification_Method": "Explicit expected sample count",
            "Status": "PASS",
        },
        {
            "Test_Family": "DYNAMIC_STEP",
            "Qualification_Method": "Duration x sample rate + initial sample",
            "Status": "PASS",
        },
        {
            "Test_Family": "DYNAMIC_RAMP",
            "Qualification_Method": "Duration x sample rate + initial sample",
            "Status": "PASS",
        },
        {
            "Test_Family": "DYNAMIC_CYCLIC",
            "Qualification_Method": "Cycle coverage and sequence integrity",
            "Status": "PASS",
        },
        {
            "Test_Family": "DYNAMIC_SINUSOIDAL",
            "Qualification_Method": "Temporal and period-frequency integrity",
            "Status": "PASS",
        },
        {
            "Test_Family": "STABILITY",
            "Qualification_Method": "Schedule and aging-day integrity",
            "Status": "PASS",
        },
        {
            "Test_Family": "UNCERTAINTY_EVALUATION",
            "Qualification_Method": "Design-only points",
            "Status": "NOT_APPLICABLE",
        },
    ]
)

display(family_completeness_final)

,Test_Family,Qualification_Method,Status
0,STATIC,Explicit expected sample count,PASS
1,DYNAMIC_STEP,Duration x sample rate + initial sample,PASS
2,DYNAMIC_RAMP,Duration x sample rate + initial sample,PASS
3,DYNAMIC_CYCLIC,Cycle coverage and sequence integrity,PASS
4,DYNAMIC_SINUSOIDAL,Temporal and period-frequency integrity,PASS
5,STABILITY,Schedule and aging-day integrity,PASS
6,UNCERTAINTY_EVALUATION,Design-only points,NOT_APPLICABLE


## 7. Experimental-State Consistency

### Engineering Question

Are measurement records associated with the correct experimental family and
design state defined by the controlled test matrix?

### Why This Check Matters

A structurally complete measurement record can still be invalid if it is
associated with the wrong experimental condition.

Before using measurements for static characterization, calibration, dynamic
analysis, or stability assessment, the metadata recorded in each measurement
family must remain consistent with the corresponding experimental design.

The qualification is performed using family-aware checks because static,
step, ramp, cyclic, sinusoidal, and stability experiments do not share the
same experimental-state variables.

### 7.1 Test-Family Mapping

Each measurement-producing `Test_ID` must belong to exactly one test family,
and that family must agree with the family assigned in `test_matrix.csv`.

In [154]:
# ------------------------------------------------------------
# Determine observed Test_Family for every measurement Test_ID
# ------------------------------------------------------------

measurement_family_records = []

for file_name in measurement_files:

    df = datasets[file_name]

    family_by_test = (
        df.groupby("Test_ID")
        .agg(
            Measurement_Test_Family=("Test_Family", "first"),
            Unique_Family_Values=("Test_Family", "nunique"),
        )
        .reset_index()
    )

    family_by_test["Source_File"] = file_name

    measurement_family_records.append(
        family_by_test
    )

measurement_family_df = pd.concat(
    measurement_family_records,
    ignore_index=True
)

display(measurement_family_df.head(10))

,Test_ID,Measurement_Test_Family,Unique_Family_Values,Source_File
0,DY-CYC-001,DYNAMIC_CYCLIC,1,cyclic_measurements.csv
1,DY-CYC-002,DYNAMIC_CYCLIC,1,cyclic_measurements.csv
2,DY-CYC-003,DYNAMIC_CYCLIC,1,cyclic_measurements.csv
3,DY-STEP-001,DYNAMIC_STEP,1,dynamic_step_measurements.csv
4,DY-STEP-002,DYNAMIC_STEP,1,dynamic_step_measurements.csv
5,DY-STEP-003,DYNAMIC_STEP,1,dynamic_step_measurements.csv
6,DY-STEP-004,DYNAMIC_STEP,1,dynamic_step_measurements.csv
7,DY-STEP-005,DYNAMIC_STEP,1,dynamic_step_measurements.csv
8,DY-STEP-006,DYNAMIC_STEP,1,dynamic_step_measurements.csv
9,DY-STEP-007,DYNAMIC_STEP,1,dynamic_step_measurements.csv


In [155]:
# ------------------------------------------------------------
# Compare measurement Test_Family with experimental design
# ------------------------------------------------------------

matrix_family_df = test_matrix_df[
    ["Test_ID", "Test_Family"]
].rename(
    columns={
        "Test_Family": "Matrix_Test_Family"
    }
)

family_check_df = measurement_family_df.merge(
    matrix_family_df,
    on="Test_ID",
    how="outer",
    indicator=True
)

family_check_df["Family_Matches"] = (
    family_check_df["Measurement_Test_Family"]
    == family_check_df["Matrix_Test_Family"]
)

display(family_check_df.head(10))

,Test_ID,Measurement_Test_Family,Unique_Family_Values,Source_File,Matrix_Test_Family,_merge,Family_Matches
0,CAL-001,STATIC,1.0,static_measurements.csv,STATIC,both,True
1,CAL-002,STATIC,1.0,static_measurements.csv,STATIC,both,True
2,CAL-003,STATIC,1.0,static_measurements.csv,STATIC,both,True
3,CAL-004,STATIC,1.0,static_measurements.csv,STATIC,both,True
4,CAL-005,STATIC,1.0,static_measurements.csv,STATIC,both,True
5,CAL-006,STATIC,1.0,static_measurements.csv,STATIC,both,True
6,CAL-007,STATIC,1.0,static_measurements.csv,STATIC,both,True
7,CAL-008,STATIC,1.0,static_measurements.csv,STATIC,both,True
8,CAL-009,STATIC,1.0,static_measurements.csv,STATIC,both,True
9,CAL-010,STATIC,1.0,static_measurements.csv,STATIC,both,True


In [156]:
# ------------------------------------------------------------
# Evaluate test-family mapping
# ------------------------------------------------------------

measurement_family_rows = family_check_df[
    family_check_df["Measurement_Test_Family"].notna()
].copy()

# Tests that contain more than one Test_Family label
multi_family_measurement_tests = measurement_family_rows[
    measurement_family_rows["Unique_Family_Values"] > 1
].copy()

# Measurement family does not match the test-matrix family
family_mismatches = measurement_family_rows[
    ~measurement_family_rows["Family_Matches"]
].copy()

# Measurement Test_ID cannot be traced to the test matrix
untraceable_family_tests = measurement_family_rows[
    measurement_family_rows["_merge"] != "both"
].copy()

test_family_mapping_pass = (
    len(multi_family_measurement_tests) == 0
    and len(family_mismatches) == 0
    and len(untraceable_family_tests) == 0
)

print(
    f"Measurement Test_IDs checked        : "
    f"{len(measurement_family_rows)}"
)

print(
    f"Tests with multiple family labels   : "
    f"{len(multi_family_measurement_tests)}"
)

print(
    f"Measurement/matrix family mismatch  : "
    f"{len(family_mismatches)}"
)

print(
    f"Untraceable measurement families    : "
    f"{len(untraceable_family_tests)}"
)

print(
    f"\nTest-family mapping qualification    : "
    f"{'PASS' if test_family_mapping_pass else 'FAIL'}"
)

Measurement Test_IDs checked        : 329
Tests with multiple family labels   : 0
Measurement/matrix family mismatch  : 0
Untraceable measurement families    : 0

Test-family mapping qualification    : PASS


### 7.2 Static Experimental-State Consistency

For static tests, the recorded experimental state must remain constant within
each `Test_ID` and must agree with the corresponding design definition.

The following variables are checked:

- relative-humidity setpoint
- temperature setpoint
- direction of environmental change
- day identifier
- run identifier
- chamber state

`Analysis_Role` is not repeated here because calibration, validation, and
characterization role integrity was qualified separately in Section 5.

In [157]:
# ------------------------------------------------------------
# Summarize observed static experimental state by Test_ID
# ------------------------------------------------------------

static_state_observed = (
    static_df
    .groupby("Test_ID")
    .agg(
        Observed_RH_Setpoint=("RH_Setpoint", "first"),
        Unique_RH_Setpoints=("RH_Setpoint", "nunique"),

        Observed_Temperature_Setpoint_C=(
            "Temperature_Setpoint_C", "first"
        ),
        Unique_Temperature_Setpoints=(
            "Temperature_Setpoint_C", "nunique"
        ),

        Observed_Direction=("Direction", "first"),
        Unique_Directions=("Direction", "nunique"),

        Observed_Day_ID=("Day_ID", "first"),
        Unique_Day_IDs=("Day_ID", "nunique"),

        Observed_Run_ID=("Run_ID", "first"),
        Unique_Run_IDs=("Run_ID", "nunique"),

        Observed_Chamber_State=("Chamber_State", "first"),
        Unique_Chamber_States=("Chamber_State", "nunique"),
    )
    .reset_index()
)

display(static_state_observed.head(10))

,Test_ID,Observed_RH_Setpoint,Unique_RH_Setpoints,Observed_Temperature_Setpoint_C,Unique_Temperature_Setpoints,Observed_Direction,Unique_Directions,Observed_Day_ID,Unique_Day_IDs,Observed_Run_ID,Unique_Run_IDs,Observed_Chamber_State,Unique_Chamber_States
0,CAL-001,20.0,1,15.0,1,UP,1,1,1,1,1,STABLE,1
1,CAL-002,20.0,1,15.0,1,UP,1,1,1,2,1,STABLE,1
2,CAL-003,20.0,1,15.0,1,UP,1,2,1,1,1,STABLE,1
3,CAL-004,20.0,1,15.0,1,UP,1,2,1,2,1,STABLE,1
4,CAL-005,20.0,1,15.0,1,DOWN,1,1,1,1,1,STABLE,1
5,CAL-006,20.0,1,15.0,1,DOWN,1,1,1,2,1,STABLE,1
6,CAL-007,20.0,1,15.0,1,DOWN,1,2,1,1,1,STABLE,1
7,CAL-008,20.0,1,15.0,1,DOWN,1,2,1,2,1,STABLE,1
8,CAL-009,20.0,1,25.0,1,UP,1,1,1,1,1,STABLE,1
9,CAL-010,20.0,1,25.0,1,UP,1,1,1,2,1,STABLE,1


In [158]:
# ------------------------------------------------------------
# Join observed static state to the experimental design
# ------------------------------------------------------------

static_state_design = (
    test_matrix_df[
        test_matrix_df["Test_Family"] == "STATIC"
    ][
        [
            "Test_ID",
            "RH_Setpoint",
            "Temperature_Setpoint_C",
            "Direction",
            "Day_ID",
            "Run_ID",
            "Chamber_State_Required",
        ]
    ]
    .copy()
)

static_state_check_df = static_state_design.merge(
    static_state_observed,
    on="Test_ID",
    how="outer",
    indicator=True
)

display(static_state_check_df.head(10))

,Test_ID,RH_Setpoint,Temperature_Setpoint_C,Direction,Day_ID,Run_ID,Chamber_State_Required,Observed_RH_Setpoint,Unique_RH_Setpoints,Observed_Temperature_Setpoint_C,Unique_Temperature_Setpoints,Observed_Direction,Unique_Directions,Observed_Day_ID,Unique_Day_IDs,Observed_Run_ID,Unique_Run_IDs,Observed_Chamber_State,Unique_Chamber_States,_merge
0,CAL-001,20.0,15.0,Up,1.0,1.0,STABLE,20.0,1,15.0,1,UP,1,1,1,1,1,STABLE,1,both
1,CAL-002,20.0,15.0,Up,1.0,2.0,STABLE,20.0,1,15.0,1,UP,1,1,1,2,1,STABLE,1,both
2,CAL-003,20.0,15.0,Up,2.0,1.0,STABLE,20.0,1,15.0,1,UP,1,2,1,1,1,STABLE,1,both
3,CAL-004,20.0,15.0,Up,2.0,2.0,STABLE,20.0,1,15.0,1,UP,1,2,1,2,1,STABLE,1,both
4,CAL-005,20.0,15.0,Down,1.0,1.0,STABLE,20.0,1,15.0,1,DOWN,1,1,1,1,1,STABLE,1,both
5,CAL-006,20.0,15.0,Down,1.0,2.0,STABLE,20.0,1,15.0,1,DOWN,1,1,1,2,1,STABLE,1,both
6,CAL-007,20.0,15.0,Down,2.0,1.0,STABLE,20.0,1,15.0,1,DOWN,1,2,1,1,1,STABLE,1,both
7,CAL-008,20.0,15.0,Down,2.0,2.0,STABLE,20.0,1,15.0,1,DOWN,1,2,1,2,1,STABLE,1,both
8,CAL-009,20.0,25.0,Up,1.0,1.0,STABLE,20.0,1,25.0,1,UP,1,1,1,1,1,STABLE,1,both
9,CAL-010,20.0,25.0,Up,1.0,2.0,STABLE,20.0,1,25.0,1,UP,1,1,1,2,1,STABLE,1,both


In [159]:
print(static_state_check_df.columns.tolist())

['Test_ID', 'RH_Setpoint', 'Temperature_Setpoint_C', 'Direction', 'Day_ID', 'Run_ID', 'Chamber_State_Required', 'Observed_RH_Setpoint', 'Unique_RH_Setpoints', 'Observed_Temperature_Setpoint_C', 'Unique_Temperature_Setpoints', 'Observed_Direction', 'Unique_Directions', 'Observed_Day_ID', 'Unique_Day_IDs', 'Observed_Run_ID', 'Unique_Run_IDs', 'Observed_Chamber_State', 'Unique_Chamber_States', '_merge']


In [160]:
# ------------------------------------------------------------
# Evaluate internal and design consistency for static tests
# ------------------------------------------------------------

static_state_check_df["Internal_State_Consistent"] = (
    (static_state_check_df["Unique_RH_Setpoints"] == 1)
    & (static_state_check_df["Unique_Temperature_Setpoints"] == 1)
    & (static_state_check_df["Unique_Directions"] == 1)
    & (static_state_check_df["Unique_Day_IDs"] == 1)
    & (static_state_check_df["Unique_Run_IDs"] == 1)
    & (static_state_check_df["Unique_Chamber_States"] == 1)
)

static_state_check_df["RH_Matches_Design"] = (
    static_state_check_df["RH_Setpoint"]
    == static_state_check_df["Observed_RH_Setpoint"]
)

static_state_check_df["Temperature_Matches_Design"] = (
    static_state_check_df["Temperature_Setpoint_C"]
    == static_state_check_df["Observed_Temperature_Setpoint_C"]
)

static_state_check_df["Direction_Matches_Design"] = (
    static_state_check_df["Direction"]
    .astype(str)
    .str.strip()
    .str.upper()
    ==
    static_state_check_df["Observed_Direction"]
    .astype(str)
    .str.strip()
    .str.upper()
)

static_state_check_df["Day_Matches_Design"] = (
    static_state_check_df["Day_ID"]
    == static_state_check_df["Observed_Day_ID"]
)

static_state_check_df["Run_Matches_Design"] = (
    static_state_check_df["Run_ID"]
    == static_state_check_df["Observed_Run_ID"]
)

static_state_check_df["Chamber_State_Matches_Design"] = (
    static_state_check_df["Chamber_State_Required"]
    .astype(str)
    .str.strip()
    .str.upper()
    ==
    static_state_check_df["Observed_Chamber_State"]
    .astype(str)
    .str.strip()
    .str.upper()
)

In [161]:
# ------------------------------------------------------------
# Display selected static-state qualification results
# ------------------------------------------------------------

display(
    static_state_check_df[
        [
            "Test_ID",
            "RH_Setpoint",
            "Observed_RH_Setpoint",
            "Temperature_Setpoint_C",
            "Observed_Temperature_Setpoint_C",
            "Direction",
            "Observed_Direction",
            "Day_ID",
            "Observed_Day_ID",
            "Run_ID",
            "Observed_Run_ID",
            "Chamber_State_Required",
            "Observed_Chamber_State",
            "Internal_State_Consistent",
        ]
    ].head(10)
)

,Test_ID,RH_Setpoint,Observed_RH_Setpoint,Temperature_Setpoint_C,Observed_Temperature_Setpoint_C,Direction,Observed_Direction,Day_ID,Observed_Day_ID,Run_ID,Observed_Run_ID,Chamber_State_Required,Observed_Chamber_State,Internal_State_Consistent
0,CAL-001,20.0,20.0,15.0,15.0,Up,UP,1.0,1,1.0,1,STABLE,STABLE,True
1,CAL-002,20.0,20.0,15.0,15.0,Up,UP,1.0,1,2.0,2,STABLE,STABLE,True
2,CAL-003,20.0,20.0,15.0,15.0,Up,UP,2.0,2,1.0,1,STABLE,STABLE,True
3,CAL-004,20.0,20.0,15.0,15.0,Up,UP,2.0,2,2.0,2,STABLE,STABLE,True
4,CAL-005,20.0,20.0,15.0,15.0,Down,DOWN,1.0,1,1.0,1,STABLE,STABLE,True
5,CAL-006,20.0,20.0,15.0,15.0,Down,DOWN,1.0,1,2.0,2,STABLE,STABLE,True
6,CAL-007,20.0,20.0,15.0,15.0,Down,DOWN,2.0,2,1.0,1,STABLE,STABLE,True
7,CAL-008,20.0,20.0,15.0,15.0,Down,DOWN,2.0,2,2.0,2,STABLE,STABLE,True
8,CAL-009,20.0,20.0,25.0,25.0,Up,UP,1.0,1,1.0,1,STABLE,STABLE,True
9,CAL-010,20.0,20.0,25.0,25.0,Up,UP,1.0,1,2.0,2,STABLE,STABLE,True


In [162]:
# ------------------------------------------------------------
# Summarize static experimental-state qualification
# ------------------------------------------------------------

static_state_pass = (
    (static_state_check_df["_merge"] == "both").all()
    and static_state_check_df[
        "Internal_State_Consistent"
    ].all()
    and static_state_check_df[
        "RH_Matches_Design"
    ].all()
    and static_state_check_df[
        "Temperature_Matches_Design"
    ].all()
    and static_state_check_df[
        "Direction_Matches_Design"
    ].all()
    and static_state_check_df[
        "Day_Matches_Design"
    ].all()
    and static_state_check_df[
        "Run_Matches_Design"
    ].all()
    and static_state_check_df[
        "Chamber_State_Matches_Design"
    ].all()
)

print(
    f"Static tests checked                : "
    f"{len(static_state_check_df)}"
)

print(
    f"Tests with internally varying state : "
    f"{(~static_state_check_df['Internal_State_Consistent']).sum()}"
)

print(
    f"RH-setpoint mismatches              : "
    f"{(~static_state_check_df['RH_Matches_Design']).sum()}"
)

print(
    f"Temperature-setpoint mismatches     : "
    f"{(~static_state_check_df['Temperature_Matches_Design']).sum()}"
)

print(
    f"Direction mismatches                : "
    f"{(~static_state_check_df['Direction_Matches_Design']).sum()}"
)

print(
    f"Day-ID mismatches                   : "
    f"{(~static_state_check_df['Day_Matches_Design']).sum()}"
)

print(
    f"Run-ID mismatches                   : "
    f"{(~static_state_check_df['Run_Matches_Design']).sum()}"
)

print(
    f"Chamber-state mismatches            : "
    f"{(~static_state_check_df['Chamber_State_Matches_Design']).sum()}"
)

print(
    f"\nStatic experimental-state qualification : "
    f"{'PASS' if static_state_pass else 'FAIL'}"
)

Static tests checked                : 252
Tests with internally varying state : 0
RH-setpoint mismatches              : 0
Temperature-setpoint mismatches     : 0
Direction mismatches                : 0
Day-ID mismatches                   : 0
Run-ID mismatches                   : 0
Chamber-state mismatches            : 0

Static experimental-state qualification : PASS


In [163]:
# ------------------------------------------------------------
# Show failed static experimental-state checks, if any
# ------------------------------------------------------------

static_state_failures = static_state_check_df[
    (static_state_check_df["_merge"] != "both")
    | (~static_state_check_df["Internal_State_Consistent"])
    | (~static_state_check_df["RH_Matches_Design"])
    | (~static_state_check_df["Temperature_Matches_Design"])
    | (~static_state_check_df["Direction_Matches_Design"])
    | (~static_state_check_df["Day_Matches_Design"])
    | (~static_state_check_df["Run_Matches_Design"])
    | (~static_state_check_df["Chamber_State_Matches_Design"])
]

if static_state_failures.empty:
    print("No static experimental-state failures detected.")
else:
    display(static_state_failures)

No static experimental-state failures detected.


### 7.3 Dynamic-Step Experimental-State Consistency

Dynamic-step tests intentionally contain a change in commanded relative
humidity. Therefore, experimental-state consistency cannot be evaluated using
the same constant-state rule applied to static tests.

For each dynamic-step `Test_ID`, this check verifies that:

- temperature setpoint remains constant and agrees with the test design
- direction remains constant and agrees with the test design
- repetition identifier remains constant and agrees with the test design
- the baseline command agrees with `RH_Start`
- the post-step command agrees with `RH_End`
- the command transition occurs at the designed baseline boundary
- chamber-state labeling is consistent with the baseline and transient
  segments

Sensor response to the command change is not evaluated here.

In [164]:
# ------------------------------------------------------------
# Extract dynamic-step experimental design
# ------------------------------------------------------------

step_df = datasets["dynamic_step_measurements.csv"]

step_design = (
    test_matrix_df[
        test_matrix_df["Test_Family"] == "DYNAMIC_STEP"
    ][
        [
            "Test_ID",
            "Temperature_Setpoint_C",
            "Direction",
            "RH_Start",
            "RH_End",
            "Repetition",
            "Baseline_s",
            "Transient_s",
            "Duration_s",
        ]
    ]
    .copy()
)

display(step_design.head(10))

,Test_ID,Temperature_Setpoint_C,Direction,RH_Start,RH_End,Repetition,Baseline_s,Transient_s,Duration_s
252,DY-STEP-001,15.0,Up,30.0,70.0,1.0,60.0,240.0,300.0
253,DY-STEP-002,15.0,Up,30.0,70.0,2.0,60.0,240.0,300.0
254,DY-STEP-003,15.0,Up,30.0,70.0,3.0,60.0,240.0,300.0
255,DY-STEP-004,15.0,Up,30.0,70.0,4.0,60.0,240.0,300.0
256,DY-STEP-005,15.0,Up,30.0,70.0,5.0,60.0,240.0,300.0
257,DY-STEP-006,25.0,Up,30.0,70.0,1.0,60.0,240.0,300.0
258,DY-STEP-007,25.0,Up,30.0,70.0,2.0,60.0,240.0,300.0
259,DY-STEP-008,25.0,Up,30.0,70.0,3.0,60.0,240.0,300.0
260,DY-STEP-009,25.0,Up,30.0,70.0,4.0,60.0,240.0,300.0
261,DY-STEP-010,25.0,Up,30.0,70.0,5.0,60.0,240.0,300.0


In [165]:
# ------------------------------------------------------------
# Summarize dynamic-step metadata by Test_ID
# ------------------------------------------------------------

step_observed = (
    step_df
    .groupby("Test_ID")
    .agg(
        Observed_Temperature_Setpoint_C=(
            "Temperature_Setpoint_C",
            "first"
        ),
        Unique_Temperature_Setpoints=(
            "Temperature_Setpoint_C",
            "nunique"
        ),

        Observed_Direction=(
            "Direction",
            "first"
        ),
        Unique_Directions=(
            "Direction",
            "nunique"
        ),

        Observed_Repetition=(
            "Repetition",
            "first"
        ),
        Unique_Repetitions=(
            "Repetition",
            "nunique"
        ),

        Observed_Command_Levels=(
            "RH_Command",
            "nunique"
        ),

        Start_Time_s=(
            "Elapsed_s",
            "min"
        ),
        End_Time_s=(
            "Elapsed_s",
            "max"
        ),
    )
    .reset_index()
)

step_state_check_df = step_design.merge(
    step_observed,
    on="Test_ID",
    how="outer",
    indicator=True
)

display(step_state_check_df.head(10))

,Test_ID,Temperature_Setpoint_C,Direction,RH_Start,RH_End,Repetition,Baseline_s,Transient_s,Duration_s,Observed_Temperature_Setpoint_C,Unique_Temperature_Setpoints,Observed_Direction,Unique_Directions,Observed_Repetition,Unique_Repetitions,Observed_Command_Levels,Start_Time_s,End_Time_s,_merge
0,DY-STEP-001,15.0,Up,30.0,70.0,1.0,60.0,240.0,300.0,15.0,1,UP,1,1,1,2,0.0,300.0,both
1,DY-STEP-002,15.0,Up,30.0,70.0,2.0,60.0,240.0,300.0,15.0,1,UP,1,2,1,2,0.0,300.0,both
2,DY-STEP-003,15.0,Up,30.0,70.0,3.0,60.0,240.0,300.0,15.0,1,UP,1,3,1,2,0.0,300.0,both
3,DY-STEP-004,15.0,Up,30.0,70.0,4.0,60.0,240.0,300.0,15.0,1,UP,1,4,1,2,0.0,300.0,both
4,DY-STEP-005,15.0,Up,30.0,70.0,5.0,60.0,240.0,300.0,15.0,1,UP,1,5,1,2,0.0,300.0,both
5,DY-STEP-006,25.0,Up,30.0,70.0,1.0,60.0,240.0,300.0,25.0,1,UP,1,1,1,2,0.0,300.0,both
6,DY-STEP-007,25.0,Up,30.0,70.0,2.0,60.0,240.0,300.0,25.0,1,UP,1,2,1,2,0.0,300.0,both
7,DY-STEP-008,25.0,Up,30.0,70.0,3.0,60.0,240.0,300.0,25.0,1,UP,1,3,1,2,0.0,300.0,both
8,DY-STEP-009,25.0,Up,30.0,70.0,4.0,60.0,240.0,300.0,25.0,1,UP,1,4,1,2,0.0,300.0,both
9,DY-STEP-010,25.0,Up,30.0,70.0,5.0,60.0,240.0,300.0,25.0,1,UP,1,5,1,2,0.0,300.0,both


In [166]:
# ------------------------------------------------------------
# Evaluate constant metadata for dynamic-step tests
# ------------------------------------------------------------

step_state_check_df["Metadata_Internal_Consistency"] = (
    (step_state_check_df["Unique_Temperature_Setpoints"] == 1)
    & (step_state_check_df["Unique_Directions"] == 1)
    & (step_state_check_df["Unique_Repetitions"] == 1)
)

step_state_check_df["Temperature_Matches_Design"] = (
    step_state_check_df["Temperature_Setpoint_C"]
    == step_state_check_df["Observed_Temperature_Setpoint_C"]
)

step_state_check_df["Direction_Matches_Design"] = (
    step_state_check_df["Direction"]
    .astype(str)
    .str.strip()
    .str.upper()
    ==
    step_state_check_df["Observed_Direction"]
    .astype(str)
    .str.strip()
    .str.upper()
)

step_state_check_df["Repetition_Matches_Design"] = (
    step_state_check_df["Repetition"]
    == step_state_check_df["Observed_Repetition"]
)

step_state_check_df["Duration_Matches_Design"] = (
    step_state_check_df["End_Time_s"]
    == step_state_check_df["Duration_s"]
)

display(
    step_state_check_df[
        [
            "Test_ID",
            "Observed_Command_Levels",
            "Metadata_Internal_Consistency",
            "Temperature_Matches_Design",
            "Direction_Matches_Design",
            "Repetition_Matches_Design",
            "Duration_Matches_Design",
        ]
    ].head(10)
)

,Test_ID,Observed_Command_Levels,Metadata_Internal_Consistency,Temperature_Matches_Design,Direction_Matches_Design,Repetition_Matches_Design,Duration_Matches_Design
0,DY-STEP-001,2,True,True,True,True,True
1,DY-STEP-002,2,True,True,True,True,True
2,DY-STEP-003,2,True,True,True,True,True
3,DY-STEP-004,2,True,True,True,True,True
4,DY-STEP-005,2,True,True,True,True,True
5,DY-STEP-006,2,True,True,True,True,True
6,DY-STEP-007,2,True,True,True,True,True
7,DY-STEP-008,2,True,True,True,True,True
8,DY-STEP-009,2,True,True,True,True,True
9,DY-STEP-010,2,True,True,True,True,True


In [167]:
# ------------------------------------------------------------
# Attach step-design parameters to every measurement row
# ------------------------------------------------------------

step_row_check_df = step_df.merge(
    step_design,
    on="Test_ID",
    how="left",
    suffixes=("_Measurement", "_Design")
)

# Before the designed baseline boundary:
# RH command must equal RH_Start.
#
# At and after the boundary:
# RH command must equal RH_End.

step_row_check_df["Expected_RH_Command"] = (
    step_row_check_df["RH_Start"]
)

post_step_mask = (
    step_row_check_df["Elapsed_s"]
    >= step_row_check_df["Baseline_s"]
)

step_row_check_df.loc[
    post_step_mask,
    "Expected_RH_Command"
] = step_row_check_df.loc[
    post_step_mask,
    "RH_End"
]

step_row_check_df["Command_Matches_Design"] = (
    step_row_check_df["RH_Command"]
    == step_row_check_df["Expected_RH_Command"]
)

display(
    step_row_check_df[
        [
            "Test_ID",
            "Elapsed_s",
            "RH_Start",
            "RH_End",
            "Baseline_s",
            "RH_Command",
            "Expected_RH_Command",
            "Command_Matches_Design",
        ]
    ].head(65)
)

,Test_ID,Elapsed_s,RH_Start,RH_End,Baseline_s,RH_Command,Expected_RH_Command,Command_Matches_Design
0,DY-STEP-001,0.0,30.0,70.0,60.0,30.0,30.0,True
1,DY-STEP-001,1.0,30.0,70.0,60.0,30.0,30.0,True
2,DY-STEP-001,2.0,30.0,70.0,60.0,30.0,30.0,True
3,DY-STEP-001,3.0,30.0,70.0,60.0,30.0,30.0,True
4,DY-STEP-001,4.0,30.0,70.0,60.0,30.0,30.0,True
...,...,...,...,...,...,...,...,...
60,DY-STEP-001,60.0,30.0,70.0,60.0,70.0,70.0,True
61,DY-STEP-001,61.0,30.0,70.0,60.0,70.0,70.0,True
62,DY-STEP-001,62.0,30.0,70.0,60.0,70.0,70.0,True
63,DY-STEP-001,63.0,30.0,70.0,60.0,70.0,70.0,True


In [168]:
# ------------------------------------------------------------
# Verify chamber-state labeling relative to step timing
# ------------------------------------------------------------

step_row_check_df["Expected_Chamber_State"] = "STABLE"

step_row_check_df.loc[
    post_step_mask,
    "Expected_Chamber_State"
] = "TRANSIENT"

step_row_check_df["Chamber_State_Matches_Segment"] = (
    step_row_check_df["Chamber_State"]
    .astype(str)
    .str.strip()
    .str.upper()
    ==
    step_row_check_df["Expected_Chamber_State"]
    .astype(str)
    .str.strip()
    .str.upper()
)

In [169]:
# ------------------------------------------------------------
# Summarize step command and chamber-state integrity by Test_ID
# ------------------------------------------------------------

step_command_summary = (
    step_row_check_df
    .groupby("Test_ID")
    .agg(
        Command_Mismatches=(
            "Command_Matches_Design",
            lambda x: (~x).sum()
        ),
        Chamber_State_Mismatches=(
            "Chamber_State_Matches_Segment",
            lambda x: (~x).sum()
        ),
    )
    .reset_index()
)

display(step_command_summary)

,Test_ID,Command_Mismatches,Chamber_State_Mismatches
0,DY-STEP-001,0,0
1,DY-STEP-002,0,0
2,DY-STEP-003,0,0
3,DY-STEP-004,0,0
4,DY-STEP-005,0,0
5,DY-STEP-006,0,0
6,DY-STEP-007,0,0
7,DY-STEP-008,0,0
8,DY-STEP-009,0,0
9,DY-STEP-010,0,0


In [170]:
# ------------------------------------------------------------
# Final dynamic-step experimental-state qualification
# ------------------------------------------------------------

step_command_pass = (
    step_command_summary["Command_Mismatches"] == 0
).all()

step_chamber_state_pass = (
    step_command_summary["Chamber_State_Mismatches"] == 0
).all()

step_metadata_pass = (
    (step_state_check_df["_merge"] == "both").all()
    and step_state_check_df[
        "Metadata_Internal_Consistency"
    ].all()
    and step_state_check_df[
        "Temperature_Matches_Design"
    ].all()
    and step_state_check_df[
        "Direction_Matches_Design"
    ].all()
    and step_state_check_df[
        "Repetition_Matches_Design"
    ].all()
    and step_state_check_df[
        "Duration_Matches_Design"
    ].all()
)

dynamic_step_state_pass = (
    step_metadata_pass
    and step_command_pass
    and step_chamber_state_pass
)

print(
    f"Dynamic-step tests checked          : "
    f"{len(step_state_check_df)}"
)

print(
    f"Metadata/design failures            : "
    f"{(~step_state_check_df['Metadata_Internal_Consistency']).sum()}"
)

print(
    f"Temperature mismatches              : "
    f"{(~step_state_check_df['Temperature_Matches_Design']).sum()}"
)

print(
    f"Direction mismatches                : "
    f"{(~step_state_check_df['Direction_Matches_Design']).sum()}"
)

print(
    f"Repetition mismatches               : "
    f"{(~step_state_check_df['Repetition_Matches_Design']).sum()}"
)

print(
    f"Duration mismatches                 : "
    f"{(~step_state_check_df['Duration_Matches_Design']).sum()}"
)

print(
    f"Command mismatches                  : "
    f"{step_command_summary['Command_Mismatches'].sum()}"
)

print(
    f"Chamber-state segment mismatches    : "
    f"{step_command_summary['Chamber_State_Mismatches'].sum()}"
)

print(
    f"\nDynamic-step state qualification    : "
    f"{'PASS' if dynamic_step_state_pass else 'FAIL'}"
)

Dynamic-step tests checked          : 30
Metadata/design failures            : 0
Temperature mismatches              : 0
Direction mismatches                : 0
Repetition mismatches               : 0
Duration mismatches                 : 0
Command mismatches                  : 0
Chamber-state segment mismatches    : 0

Dynamic-step state qualification    : PASS


### 7.4 Dynamic-Ramp Experimental-State Consistency

Dynamic-ramp tests intentionally vary the commanded relative humidity
continuously over time.

For each ramp `Test_ID`, this check verifies that:

- temperature setpoint remains constant and agrees with the experimental design
- direction remains constant and agrees with the design
- repetition identifier remains constant and agrees with the design
- test duration agrees with the design
- the observed command follows the expected linear trajectory from
  `RH_Start` to `RH_End`
- the commanded ramp rate is consistent with the designed ramp rate

Sensor tracking error is not evaluated here. This section verifies the
integrity of the commanded experiment, not DUT performance.

In [171]:
# ------------------------------------------------------------
# Extract dynamic-ramp experimental design
# ------------------------------------------------------------

ramp_df = datasets["ramp_measurements.csv"]

ramp_design = (
    test_matrix_df[
        test_matrix_df["Test_Family"] == "DYNAMIC_RAMP"
    ][
        [
            "Test_ID",
            "Temperature_Setpoint_C",
            "Direction",
            "RH_Start",
            "RH_End",
            "Repetition",
            "Duration_s",
            "Ramp_Rate_RH_per_min",
        ]
    ]
    .copy()
)

display(ramp_design.head(10))

,Test_ID,Temperature_Setpoint_C,Direction,RH_Start,RH_End,Repetition,Duration_s,Ramp_Rate_RH_per_min
282,DY-RAMP-001,15.0,Up,30.0,70.0,1.0,1200.0,2.0
283,DY-RAMP-002,15.0,Up,30.0,70.0,2.0,1200.0,2.0
284,DY-RAMP-003,15.0,Up,30.0,70.0,3.0,1200.0,2.0
285,DY-RAMP-004,25.0,Up,30.0,70.0,1.0,1200.0,2.0
286,DY-RAMP-005,25.0,Up,30.0,70.0,2.0,1200.0,2.0
287,DY-RAMP-006,25.0,Up,30.0,70.0,3.0,1200.0,2.0
288,DY-RAMP-007,35.0,Up,30.0,70.0,1.0,1200.0,2.0
289,DY-RAMP-008,35.0,Up,30.0,70.0,2.0,1200.0,2.0
290,DY-RAMP-009,35.0,Up,30.0,70.0,3.0,1200.0,2.0
291,DY-RAMP-010,15.0,Down,70.0,30.0,1.0,1200.0,2.0


In [172]:
# ------------------------------------------------------------
# Summarize observed ramp metadata by Test_ID
# ------------------------------------------------------------

ramp_observed = (
    ramp_df
    .groupby("Test_ID")
    .agg(
        Observed_Temperature_Setpoint_C=(
            "Temperature_Setpoint_C", "first"
        ),
        Unique_Temperature_Setpoints=(
            "Temperature_Setpoint_C", "nunique"
        ),

        Observed_Direction=("Direction", "first"),
        Unique_Directions=("Direction", "nunique"),

        Observed_Repetition=("Repetition", "first"),
        Unique_Repetitions=("Repetition", "nunique"),

        Observed_Chamber_State=("Chamber_State", "first"),
        Unique_Chamber_States=("Chamber_State", "nunique"),

        Start_Time_s=("Elapsed_s", "min"),
        End_Time_s=("Elapsed_s", "max"),

        Observed_Start_Command=("RH_Command", "first"),
        Observed_End_Command=("RH_Command", "last"),
    )
    .reset_index()
)

ramp_state_check_df = ramp_design.merge(
    ramp_observed,
    on="Test_ID",
    how="outer",
    indicator=True
)

display(ramp_state_check_df.head(10))

,Test_ID,Temperature_Setpoint_C,Direction,RH_Start,RH_End,Repetition,Duration_s,Ramp_Rate_RH_per_min,Observed_Temperature_Setpoint_C,Unique_Temperature_Setpoints,...,Unique_Directions,Observed_Repetition,Unique_Repetitions,Observed_Chamber_State,Unique_Chamber_States,Start_Time_s,End_Time_s,Observed_Start_Command,Observed_End_Command,_merge
0,DY-RAMP-001,15.0,Up,30.0,70.0,1.0,1200.0,2.0,15.0,1,...,1,1,1,TRANSIENT,1,0.0,1200.0,30.0,70.0,both
1,DY-RAMP-002,15.0,Up,30.0,70.0,2.0,1200.0,2.0,15.0,1,...,1,2,1,TRANSIENT,1,0.0,1200.0,30.0,70.0,both
2,DY-RAMP-003,15.0,Up,30.0,70.0,3.0,1200.0,2.0,15.0,1,...,1,3,1,TRANSIENT,1,0.0,1200.0,30.0,70.0,both
3,DY-RAMP-004,25.0,Up,30.0,70.0,1.0,1200.0,2.0,25.0,1,...,1,1,1,TRANSIENT,1,0.0,1200.0,30.0,70.0,both
4,DY-RAMP-005,25.0,Up,30.0,70.0,2.0,1200.0,2.0,25.0,1,...,1,2,1,TRANSIENT,1,0.0,1200.0,30.0,70.0,both
5,DY-RAMP-006,25.0,Up,30.0,70.0,3.0,1200.0,2.0,25.0,1,...,1,3,1,TRANSIENT,1,0.0,1200.0,30.0,70.0,both
6,DY-RAMP-007,35.0,Up,30.0,70.0,1.0,1200.0,2.0,35.0,1,...,1,1,1,TRANSIENT,1,0.0,1200.0,30.0,70.0,both
7,DY-RAMP-008,35.0,Up,30.0,70.0,2.0,1200.0,2.0,35.0,1,...,1,2,1,TRANSIENT,1,0.0,1200.0,30.0,70.0,both
8,DY-RAMP-009,35.0,Up,30.0,70.0,3.0,1200.0,2.0,35.0,1,...,1,3,1,TRANSIENT,1,0.0,1200.0,30.0,70.0,both
9,DY-RAMP-010,15.0,Down,70.0,30.0,1.0,1200.0,2.0,15.0,1,...,1,1,1,TRANSIENT,1,0.0,1200.0,70.0,30.0,both


In [173]:
# ------------------------------------------------------------
# Evaluate constant metadata and endpoint consistency
# ------------------------------------------------------------

import numpy as np

ramp_state_check_df["Metadata_Internal_Consistency"] = (
    (ramp_state_check_df["Unique_Temperature_Setpoints"] == 1)
    & (ramp_state_check_df["Unique_Directions"] == 1)
    & (ramp_state_check_df["Unique_Repetitions"] == 1)
    & (ramp_state_check_df["Unique_Chamber_States"] == 1)
)

ramp_state_check_df["Temperature_Matches_Design"] = (
    ramp_state_check_df["Temperature_Setpoint_C"]
    == ramp_state_check_df["Observed_Temperature_Setpoint_C"]
)

ramp_state_check_df["Direction_Matches_Design"] = (
    ramp_state_check_df["Direction"]
    .astype(str)
    .str.strip()
    .str.upper()
    ==
    ramp_state_check_df["Observed_Direction"]
    .astype(str)
    .str.strip()
    .str.upper()
)

ramp_state_check_df["Repetition_Matches_Design"] = (
    ramp_state_check_df["Repetition"]
    == ramp_state_check_df["Observed_Repetition"]
)

ramp_state_check_df["Duration_Matches_Design"] = (
    ramp_state_check_df["End_Time_s"]
    == ramp_state_check_df["Duration_s"]
)

ramp_state_check_df["Start_Command_Matches"] = np.isclose(
    ramp_state_check_df["Observed_Start_Command"],
    ramp_state_check_df["RH_Start"],
    atol=1e-6
)

ramp_state_check_df["End_Command_Matches"] = np.isclose(
    ramp_state_check_df["Observed_End_Command"],
    ramp_state_check_df["RH_End"],
    atol=1e-6
)

display(
    ramp_state_check_df[
        [
            "Test_ID",
            "RH_Start",
            "Observed_Start_Command",
            "RH_End",
            "Observed_End_Command",
            "Duration_s",
            "End_Time_s",
            "Metadata_Internal_Consistency",
        ]
    ].head(10)
)

,Test_ID,RH_Start,Observed_Start_Command,RH_End,Observed_End_Command,Duration_s,End_Time_s,Metadata_Internal_Consistency
0,DY-RAMP-001,30.0,30.0,70.0,70.0,1200.0,1200.0,True
1,DY-RAMP-002,30.0,30.0,70.0,70.0,1200.0,1200.0,True
2,DY-RAMP-003,30.0,30.0,70.0,70.0,1200.0,1200.0,True
3,DY-RAMP-004,30.0,30.0,70.0,70.0,1200.0,1200.0,True
4,DY-RAMP-005,30.0,30.0,70.0,70.0,1200.0,1200.0,True
5,DY-RAMP-006,30.0,30.0,70.0,70.0,1200.0,1200.0,True
6,DY-RAMP-007,30.0,30.0,70.0,70.0,1200.0,1200.0,True
7,DY-RAMP-008,30.0,30.0,70.0,70.0,1200.0,1200.0,True
8,DY-RAMP-009,30.0,30.0,70.0,70.0,1200.0,1200.0,True
9,DY-RAMP-010,70.0,70.0,30.0,30.0,1200.0,1200.0,True


In [174]:
# ------------------------------------------------------------
# Attach ramp design to every measurement record
# ------------------------------------------------------------

ramp_row_check_df = ramp_df.merge(
    ramp_design,
    on="Test_ID",
    how="left",
    suffixes=("_Measurement", "_Design")
)

ramp_row_check_df["Expected_RH_Command"] = (
    ramp_row_check_df["RH_Start"]
    +
    (
        ramp_row_check_df["RH_End"]
        - ramp_row_check_df["RH_Start"]
    )
    *
    (
        ramp_row_check_df["Elapsed_s"]
        / ramp_row_check_df["Duration_s"]
    )
)

ramp_row_check_df["Command_Matches_Design"] = np.isclose(
    ramp_row_check_df["RH_Command"],
    ramp_row_check_df["Expected_RH_Command"],
    atol=1e-6
)

display(
    ramp_row_check_df[
        [
            "Test_ID",
            "Elapsed_s",
            "RH_Start",
            "RH_End",
            "RH_Command",
            "Expected_RH_Command",
            "Command_Matches_Design",
        ]
    ].head(15)
)

,Test_ID,Elapsed_s,RH_Start,RH_End,RH_Command,Expected_RH_Command,Command_Matches_Design
0,DY-RAMP-001,0.0,30.0,70.0,30.000000,30.000000,True
1,DY-RAMP-001,1.0,30.0,70.0,30.033333,30.033333,True
2,DY-RAMP-001,2.0,30.0,70.0,30.066667,30.066667,True
3,DY-RAMP-001,3.0,30.0,70.0,30.100000,30.100000,True
4,DY-RAMP-001,4.0,30.0,70.0,30.133333,30.133333,True
5,DY-RAMP-001,5.0,30.0,70.0,30.166667,30.166667,True
6,DY-RAMP-001,6.0,30.0,70.0,30.200000,30.200000,True
7,DY-RAMP-001,7.0,30.0,70.0,30.233333,30.233333,True
8,DY-RAMP-001,8.0,30.0,70.0,30.266667,30.266667,True
9,DY-RAMP-001,9.0,30.0,70.0,30.300000,30.300000,True


In [175]:
# ------------------------------------------------------------
# Verify commanded ramp rate against experimental design
# ------------------------------------------------------------

ramp_state_check_df["Observed_Ramp_Rate_RH_per_min"] = (
    (
        ramp_state_check_df["Observed_End_Command"]
        - ramp_state_check_df["Observed_Start_Command"]
    )
    /
    (
        ramp_state_check_df["Duration_s"] / 60.0
    )
)

# The design stores ramp-rate magnitude.
ramp_state_check_df["Ramp_Rate_Matches_Design"] = np.isclose(
    ramp_state_check_df[
        "Observed_Ramp_Rate_RH_per_min"
    ].abs(),
    ramp_state_check_df["Ramp_Rate_RH_per_min"].abs(),
    atol=1e-6
)

display(
    ramp_state_check_df[
        [
            "Test_ID",
            "Direction",
            "Ramp_Rate_RH_per_min",
            "Observed_Ramp_Rate_RH_per_min",
            "Ramp_Rate_Matches_Design",
        ]
    ]
)

,Test_ID,Direction,Ramp_Rate_RH_per_min,Observed_Ramp_Rate_RH_per_min,Ramp_Rate_Matches_Design
0,DY-RAMP-001,Up,2.0,2.0,True
1,DY-RAMP-002,Up,2.0,2.0,True
2,DY-RAMP-003,Up,2.0,2.0,True
3,DY-RAMP-004,Up,2.0,2.0,True
4,DY-RAMP-005,Up,2.0,2.0,True
5,DY-RAMP-006,Up,2.0,2.0,True
6,DY-RAMP-007,Up,2.0,2.0,True
7,DY-RAMP-008,Up,2.0,2.0,True
8,DY-RAMP-009,Up,2.0,2.0,True
9,DY-RAMP-010,Down,2.0,-2.0,True


In [176]:
# ------------------------------------------------------------
# Summarize commanded-ramp trajectory integrity
# ------------------------------------------------------------

ramp_command_summary = (
    ramp_row_check_df
    .groupby("Test_ID")
    .agg(
        Command_Mismatches=(
            "Command_Matches_Design",
            lambda x: (~x).sum()
        )
    )
    .reset_index()
)

display(ramp_command_summary)

,Test_ID,Command_Mismatches
0,DY-RAMP-001,0
1,DY-RAMP-002,0
2,DY-RAMP-003,0
3,DY-RAMP-004,0
4,DY-RAMP-005,0
5,DY-RAMP-006,0
6,DY-RAMP-007,0
7,DY-RAMP-008,0
8,DY-RAMP-009,0
9,DY-RAMP-010,0


In [177]:
# ------------------------------------------------------------
# Final dynamic-ramp experimental-state qualification
# ------------------------------------------------------------

ramp_command_pass = (
    ramp_command_summary["Command_Mismatches"] == 0
).all()

ramp_metadata_pass = (
    (ramp_state_check_df["_merge"] == "both").all()
    and ramp_state_check_df[
        "Metadata_Internal_Consistency"
    ].all()
    and ramp_state_check_df[
        "Temperature_Matches_Design"
    ].all()
    and ramp_state_check_df[
        "Direction_Matches_Design"
    ].all()
    and ramp_state_check_df[
        "Repetition_Matches_Design"
    ].all()
    and ramp_state_check_df[
        "Duration_Matches_Design"
    ].all()
    and ramp_state_check_df[
        "Start_Command_Matches"
    ].all()
    and ramp_state_check_df[
        "End_Command_Matches"
    ].all()
    and ramp_state_check_df[
        "Ramp_Rate_Matches_Design"
    ].all()
)

dynamic_ramp_state_pass = (
    ramp_metadata_pass
    and ramp_command_pass
)

print(
    f"Dynamic-ramp tests checked          : "
    f"{len(ramp_state_check_df)}"
)

print(
    f"Internally varying metadata         : "
    f"{(~ramp_state_check_df['Metadata_Internal_Consistency']).sum()}"
)

print(
    f"Temperature mismatches              : "
    f"{(~ramp_state_check_df['Temperature_Matches_Design']).sum()}"
)

print(
    f"Direction mismatches                : "
    f"{(~ramp_state_check_df['Direction_Matches_Design']).sum()}"
)

print(
    f"Repetition mismatches               : "
    f"{(~ramp_state_check_df['Repetition_Matches_Design']).sum()}"
)

print(
    f"Duration mismatches                 : "
    f"{(~ramp_state_check_df['Duration_Matches_Design']).sum()}"
)

print(
    f"Start-command mismatches            : "
    f"{(~ramp_state_check_df['Start_Command_Matches']).sum()}"
)

print(
    f"End-command mismatches              : "
    f"{(~ramp_state_check_df['End_Command_Matches']).sum()}"
)

print(
    f"Ramp-rate mismatches                : "
    f"{(~ramp_state_check_df['Ramp_Rate_Matches_Design']).sum()}"
)

print(
    f"Command-trajectory mismatches       : "
    f"{ramp_command_summary['Command_Mismatches'].sum()}"
)

print(
    f"\nDynamic-ramp state qualification    : "
    f"{'PASS' if dynamic_ramp_state_pass else 'FAIL'}"
)

Dynamic-ramp tests checked          : 18
Internally varying metadata         : 0
Temperature mismatches              : 0
Direction mismatches                : 0
Repetition mismatches               : 0
Duration mismatches                 : 0
Start-command mismatches            : 0
End-command mismatches              : 0
Ramp-rate mismatches                : 0
Command-trajectory mismatches       : 0

Dynamic-ramp state qualification    : PASS


### 7.5 Dynamic-Cyclic Experimental-State Consistency

Cyclic tests repeatedly exercise the sensor between defined low and high
relative-humidity limits.

The experimental design specifies:

- temperature setpoint
- low and high RH command limits
- session identifier
- number of cycles

Because the test matrix does not define a cycle period or cycle-specific ramp
rate, no unsupported waveform-shape assumption is introduced during dataset
qualification.

This check verifies that:

- temperature remains consistent with the design
- session identity agrees with the design
- the required number of cycles is present
- cycle indices form the intended sequence
- each cycle reaches the designed low and high RH limits
- commanded RH remains within the designed range

Dynamic hysteresis, tracking error, and cycle-to-cycle sensor performance are
not evaluated in DQ-001.

In [178]:
# ------------------------------------------------------------
# Extract dynamic-cyclic experimental design
# ------------------------------------------------------------

cyclic_df = datasets["cyclic_measurements.csv"]

cyclic_design = (
    test_matrix_df[
        test_matrix_df["Test_Family"] == "DYNAMIC_CYCLIC"
    ][
        [
            "Test_ID",
            "Temperature_Setpoint_C",
            "RH_Low",
            "RH_High",
            "Session",
            "Cycle_Count",
        ]
    ]
    .copy()
)

display(cyclic_design)

,Test_ID,Temperature_Setpoint_C,RH_Low,RH_High,Session,Cycle_Count
300,DY-CYC-001,25.0,30.0,70.0,1.0,5.0
301,DY-CYC-002,25.0,30.0,70.0,2.0,5.0
302,DY-CYC-003,25.0,30.0,70.0,3.0,5.0


In [179]:
# ------------------------------------------------------------
# Summarize observed cyclic metadata by Test_ID
# ------------------------------------------------------------

cyclic_state_observed = (
    cyclic_df
    .groupby("Test_ID")
    .agg(
        Observed_Temperature_Setpoint_C=(
            "Temperature_Setpoint_C", "first"
        ),
        Unique_Temperature_Setpoints=(
            "Temperature_Setpoint_C", "nunique"
        ),

        Observed_Session=("Session", "first"),
        Unique_Sessions=("Session", "nunique"),

        Observed_Cycles=("Cycle_Index", "nunique"),
        Min_Cycle_Index=("Cycle_Index", "min"),
        Max_Cycle_Index=("Cycle_Index", "max"),

        Minimum_RH_Command=("RH_Command", "min"),
        Maximum_RH_Command=("RH_Command", "max"),
    )
    .reset_index()
)

cyclic_state_check_df = cyclic_design.merge(
    cyclic_state_observed,
    on="Test_ID",
    how="outer",
    indicator=True
)

display(cyclic_state_check_df)

,Test_ID,Temperature_Setpoint_C,RH_Low,RH_High,Session,Cycle_Count,Observed_Temperature_Setpoint_C,Unique_Temperature_Setpoints,Observed_Session,Unique_Sessions,Observed_Cycles,Min_Cycle_Index,Max_Cycle_Index,Minimum_RH_Command,Maximum_RH_Command,_merge
0,DY-CYC-001,25.0,30.0,70.0,1.0,5.0,25.0,1,1,1,5,1,5,30.0,70.0,both
1,DY-CYC-002,25.0,30.0,70.0,2.0,5.0,25.0,1,2,1,5,1,5,30.0,70.0,both
2,DY-CYC-003,25.0,30.0,70.0,3.0,5.0,25.0,1,3,1,5,1,5,30.0,70.0,both


In [180]:
# ------------------------------------------------------------
# Evaluate cyclic metadata and overall command-range integrity
# ------------------------------------------------------------

import numpy as np

cyclic_state_check_df["Temperature_Internal_Consistency"] = (
    cyclic_state_check_df["Unique_Temperature_Setpoints"] == 1
)

cyclic_state_check_df["Session_Internal_Consistency"] = (
    cyclic_state_check_df["Unique_Sessions"] == 1
)

cyclic_state_check_df["Temperature_Matches_Design"] = (
    cyclic_state_check_df["Temperature_Setpoint_C"]
    == cyclic_state_check_df["Observed_Temperature_Setpoint_C"]
)

cyclic_state_check_df["Session_Matches_Design"] = (
    cyclic_state_check_df["Session"]
    == cyclic_state_check_df["Observed_Session"]
)

cyclic_state_check_df["Cycle_Count_Matches_Design"] = (
    cyclic_state_check_df["Cycle_Count"]
    == cyclic_state_check_df["Observed_Cycles"]
)

cyclic_state_check_df["Cycle_Index_Sequence_Valid"] = (
    (cyclic_state_check_df["Min_Cycle_Index"] == 1)
    &
    (
        cyclic_state_check_df["Max_Cycle_Index"]
        == cyclic_state_check_df["Cycle_Count"]
    )
)

cyclic_state_check_df["Low_Command_Matches_Design"] = np.isclose(
    cyclic_state_check_df["Minimum_RH_Command"],
    cyclic_state_check_df["RH_Low"],
    atol=1e-6
)

cyclic_state_check_df["High_Command_Matches_Design"] = np.isclose(
    cyclic_state_check_df["Maximum_RH_Command"],
    cyclic_state_check_df["RH_High"],
    atol=1e-6
)

display(
    cyclic_state_check_df[
        [
            "Test_ID",
            "Session",
            "Observed_Session",
            "Cycle_Count",
            "Observed_Cycles",
            "Min_Cycle_Index",
            "Max_Cycle_Index",
            "RH_Low",
            "Minimum_RH_Command",
            "RH_High",
            "Maximum_RH_Command",
        ]
    ]
)

,Test_ID,Session,Observed_Session,Cycle_Count,Observed_Cycles,Min_Cycle_Index,Max_Cycle_Index,RH_Low,Minimum_RH_Command,RH_High,Maximum_RH_Command
0,DY-CYC-001,1.0,1,5.0,5,1,5,30.0,30.0,70.0,70.0
1,DY-CYC-002,2.0,2,5.0,5,1,5,30.0,30.0,70.0,70.0
2,DY-CYC-003,3.0,3,5.0,5,1,5,30.0,30.0,70.0,70.0


In [181]:
# ------------------------------------------------------------
# Evaluate RH-command coverage within each individual cycle
# ------------------------------------------------------------

cyclic_cycle_check_df = (
    cyclic_df
    .groupby(
        ["Test_ID", "Cycle_Index"]
    )
    .agg(
        Cycle_Min_RH_Command=("RH_Command", "min"),
        Cycle_Max_RH_Command=("RH_Command", "max"),
        Samples_In_Cycle=("Sample_ID", "size"),
    )
    .reset_index()
)

cyclic_cycle_check_df = cyclic_cycle_check_df.merge(
    cyclic_design[
        [
            "Test_ID",
            "RH_Low",
            "RH_High",
        ]
    ],
    on="Test_ID",
    how="left"
)

cyclic_cycle_check_df["Low_Limit_Reached"] = np.isclose(
    cyclic_cycle_check_df["Cycle_Min_RH_Command"],
    cyclic_cycle_check_df["RH_Low"],
    atol=1e-6
)

cyclic_cycle_check_df["High_Limit_Reached"] = np.isclose(
    cyclic_cycle_check_df["Cycle_Max_RH_Command"],
    cyclic_cycle_check_df["RH_High"],
    atol=1e-6
)

display(cyclic_cycle_check_df)

,Test_ID,Cycle_Index,Cycle_Min_RH_Command,Cycle_Max_RH_Command,Samples_In_Cycle,RH_Low,RH_High,Low_Limit_Reached,High_Limit_Reached
0,DY-CYC-001,1,30.0,70.0,600,30.0,70.0,True,True
1,DY-CYC-001,2,30.0,70.0,600,30.0,70.0,True,True
2,DY-CYC-001,3,30.0,70.0,600,30.0,70.0,True,True
3,DY-CYC-001,4,30.0,70.0,600,30.0,70.0,True,True
4,DY-CYC-001,5,30.0,70.0,601,30.0,70.0,True,True
5,DY-CYC-002,1,30.0,70.0,600,30.0,70.0,True,True
6,DY-CYC-002,2,30.0,70.0,600,30.0,70.0,True,True
7,DY-CYC-002,3,30.0,70.0,600,30.0,70.0,True,True
8,DY-CYC-002,4,30.0,70.0,600,30.0,70.0,True,True
9,DY-CYC-002,5,30.0,70.0,601,30.0,70.0,True,True


In [182]:
# ------------------------------------------------------------
# Verify that all cyclic command values remain within design limits
# ------------------------------------------------------------

cyclic_row_check_df = cyclic_df.merge(
    cyclic_design[
        [
            "Test_ID",
            "RH_Low",
            "RH_High",
        ]
    ],
    on="Test_ID",
    how="left"
)

cyclic_row_check_df["Command_Within_Design_Range"] = (
    (cyclic_row_check_df["RH_Command"] >= cyclic_row_check_df["RH_Low"])
    &
    (cyclic_row_check_df["RH_Command"] <= cyclic_row_check_df["RH_High"])
)

out_of_range_commands = (
    ~cyclic_row_check_df["Command_Within_Design_Range"]
).sum()

print(
    f"Cyclic command values outside design range : "
    f"{out_of_range_commands}"
)

Cyclic command values outside design range : 0


In [183]:
# ------------------------------------------------------------
# Final dynamic-cyclic experimental-state qualification
# ------------------------------------------------------------

cycle_endpoint_pass = (
    cyclic_cycle_check_df["Low_Limit_Reached"].all()
    and cyclic_cycle_check_df["High_Limit_Reached"].all()
)

cyclic_metadata_pass = (
    (cyclic_state_check_df["_merge"] == "both").all()
    and cyclic_state_check_df[
        "Temperature_Internal_Consistency"
    ].all()
    and cyclic_state_check_df[
        "Session_Internal_Consistency"
    ].all()
    and cyclic_state_check_df[
        "Temperature_Matches_Design"
    ].all()
    and cyclic_state_check_df[
        "Session_Matches_Design"
    ].all()
    and cyclic_state_check_df[
        "Cycle_Count_Matches_Design"
    ].all()
    and cyclic_state_check_df[
        "Cycle_Index_Sequence_Valid"
    ].all()
)

dynamic_cyclic_state_pass = (
    cyclic_metadata_pass
    and cycle_endpoint_pass
    and out_of_range_commands == 0
)

print(
    f"Dynamic-cyclic tests checked          : "
    f"{len(cyclic_state_check_df)}"
)

print(
    f"Individual cycles checked             : "
    f"{len(cyclic_cycle_check_df)}"
)

print(
    f"Temperature mismatches                : "
    f"{(~cyclic_state_check_df['Temperature_Matches_Design']).sum()}"
)

print(
    f"Session mismatches                    : "
    f"{(~cyclic_state_check_df['Session_Matches_Design']).sum()}"
)

print(
    f"Cycle-count mismatches                : "
    f"{(~cyclic_state_check_df['Cycle_Count_Matches_Design']).sum()}"
)

print(
    f"Invalid cycle-index sequences         : "
    f"{(~cyclic_state_check_df['Cycle_Index_Sequence_Valid']).sum()}"
)

print(
    f"Cycles not reaching RH_Low            : "
    f"{(~cyclic_cycle_check_df['Low_Limit_Reached']).sum()}"
)

print(
    f"Cycles not reaching RH_High           : "
    f"{(~cyclic_cycle_check_df['High_Limit_Reached']).sum()}"
)

print(
    f"Out-of-range command records          : "
    f"{out_of_range_commands}"
)

print(
    f"\nDynamic-cyclic state qualification    : "
    f"{'PASS' if dynamic_cyclic_state_pass else 'FAIL'}"
)

Dynamic-cyclic tests checked          : 3
Individual cycles checked             : 15
Temperature mismatches                : 0
Session mismatches                    : 0
Cycle-count mismatches                : 0
Invalid cycle-index sequences         : 0
Cycles not reaching RH_Low            : 0
Cycles not reaching RH_High           : 0
Out-of-range command records          : 0

Dynamic-cyclic state qualification    : PASS


### 7.6 Sinusoidal Experimental-State Consistency

Sinusoidal tests apply a periodic RH excitation defined by a center value,
amplitude, period, and frequency.

For each sinusoidal `Test_ID`, this check verifies that:

- temperature setpoint remains constant and agrees with the experimental design
- center RH and amplitude are correctly represented in the commanded waveform
- recorded period and frequency agree with the experimental design
- the complete `RH_Command(t)` trajectory follows the designed sinusoidal input

The expected command is

$$
RH_{\mathrm{Command}}(t)
=
RH_{\mathrm{Center}}
+
RH_{\mathrm{Amplitude}}
\sin\left(\frac{2\pi t}{T}\right)
$$

where \(T\) is the designed sinusoidal period.

Sensor gain, phase lag, and tracking error are not evaluated in DQ-001.

In [184]:
# ------------------------------------------------------------
# Extract sinusoidal experimental design
# ------------------------------------------------------------

sinusoidal_df = datasets["sinusoidal_measurements.csv"]

sinusoidal_design = (
    test_matrix_df[
        test_matrix_df["Test_Family"] == "DYNAMIC_SINUSOIDAL"
    ][
        [
            "Test_ID",
            "Temperature_Setpoint_C",
            "RH_Center",
            "RH_Amplitude",
            "Period_s",
            "Frequency_Hz",
        ]
    ]
    .copy()
)

display(sinusoidal_design)

,Test_ID,Temperature_Setpoint_C,RH_Center,RH_Amplitude,Period_s,Frequency_Hz
303,DY-SIN-001,25.0,50.0,10.0,600.0,0.001667
304,DY-SIN-002,25.0,50.0,10.0,300.0,0.003333
305,DY-SIN-003,25.0,50.0,10.0,120.0,0.008333
306,DY-SIN-004,25.0,50.0,10.0,60.0,0.016667


In [185]:
# ------------------------------------------------------------
# Summarize observed sinusoidal metadata by Test_ID
# ------------------------------------------------------------

sinusoidal_state_observed = (
    sinusoidal_df
    .groupby("Test_ID")
    .agg(
        Observed_Temperature_Setpoint_C=(
            "Temperature_Setpoint_C", "first"
        ),
        Unique_Temperature_Setpoints=(
            "Temperature_Setpoint_C", "nunique"
        ),

        Observed_Period_s=("Period_s", "first"),
        Unique_Period_Values=("Period_s", "nunique"),

        Observed_Frequency_Hz=("Frequency_Hz", "first"),
        Unique_Frequency_Values=("Frequency_Hz", "nunique"),

        Minimum_RH_Command=("RH_Command", "min"),
        Maximum_RH_Command=("RH_Command", "max"),

        Start_Time_s=("Elapsed_s", "min"),
        End_Time_s=("Elapsed_s", "max"),
    )
    .reset_index()
)

sinusoidal_state_check_df = sinusoidal_design.merge(
    sinusoidal_state_observed,
    on="Test_ID",
    how="outer",
    indicator=True
)

display(sinusoidal_state_check_df)

,Test_ID,Temperature_Setpoint_C,RH_Center,RH_Amplitude,Period_s,Frequency_Hz,Observed_Temperature_Setpoint_C,Unique_Temperature_Setpoints,Observed_Period_s,Unique_Period_Values,Observed_Frequency_Hz,Unique_Frequency_Values,Minimum_RH_Command,Maximum_RH_Command,Start_Time_s,End_Time_s,_merge
0,DY-SIN-001,25.0,50.0,10.0,600.0,0.001667,25.0,1,600,1,0.001667,1,40.0,60.0,0.0,3000.0,both
1,DY-SIN-002,25.0,50.0,10.0,300.0,0.003333,25.0,1,300,1,0.003333,1,40.0,60.0,0.0,1500.0,both
2,DY-SIN-003,25.0,50.0,10.0,120.0,0.008333,25.0,1,120,1,0.008333,1,40.0,60.0,0.0,600.0,both
3,DY-SIN-004,25.0,50.0,10.0,60.0,0.016667,25.0,1,60,1,0.016667,1,40.0,60.0,0.0,300.0,both


In [186]:
# ------------------------------------------------------------
# Evaluate sinusoidal metadata consistency
# ------------------------------------------------------------

import numpy as np

sinusoidal_state_check_df["Metadata_Internal_Consistency"] = (
    (sinusoidal_state_check_df["Unique_Temperature_Setpoints"] == 1)
    & (sinusoidal_state_check_df["Unique_Period_Values"] == 1)
    & (sinusoidal_state_check_df["Unique_Frequency_Values"] == 1)
)

sinusoidal_state_check_df["Temperature_Matches_Design"] = (
    sinusoidal_state_check_df["Temperature_Setpoint_C"]
    == sinusoidal_state_check_df[
        "Observed_Temperature_Setpoint_C"
    ]
)

sinusoidal_state_check_df["Period_Matches_Design"] = np.isclose(
    sinusoidal_state_check_df["Period_s"],
    sinusoidal_state_check_df["Observed_Period_s"],
    atol=1e-6
)

sinusoidal_state_check_df["Frequency_Matches_Design"] = np.isclose(
    sinusoidal_state_check_df["Frequency_Hz"],
    sinusoidal_state_check_df["Observed_Frequency_Hz"],
    atol=1e-6
)

sinusoidal_state_check_df["Low_Command_Matches_Design"] = np.isclose(
    sinusoidal_state_check_df["Minimum_RH_Command"],
    (
        sinusoidal_state_check_df["RH_Center"]
        - sinusoidal_state_check_df["RH_Amplitude"]
    ),
    atol=1e-6
)

sinusoidal_state_check_df["High_Command_Matches_Design"] = np.isclose(
    sinusoidal_state_check_df["Maximum_RH_Command"],
    (
        sinusoidal_state_check_df["RH_Center"]
        + sinusoidal_state_check_df["RH_Amplitude"]
    ),
    atol=1e-6
)

display(
    sinusoidal_state_check_df[
        [
            "Test_ID",
            "RH_Center",
            "RH_Amplitude",
            "Minimum_RH_Command",
            "Maximum_RH_Command",
            "Period_s",
            "Observed_Period_s",
            "Frequency_Hz",
            "Observed_Frequency_Hz",
        ]
    ]
)

,Test_ID,RH_Center,RH_Amplitude,Minimum_RH_Command,Maximum_RH_Command,Period_s,Observed_Period_s,Frequency_Hz,Observed_Frequency_Hz
0,DY-SIN-001,50.0,10.0,40.0,60.0,600.0,600,0.001667,0.001667
1,DY-SIN-002,50.0,10.0,40.0,60.0,300.0,300,0.003333,0.003333
2,DY-SIN-003,50.0,10.0,40.0,60.0,120.0,120,0.008333,0.008333
3,DY-SIN-004,50.0,10.0,40.0,60.0,60.0,60,0.016667,0.016667


In [187]:
# ------------------------------------------------------------
# Verify the complete sinusoidal command trajectory
# ------------------------------------------------------------

sinusoidal_row_check_df = sinusoidal_df.merge(
    sinusoidal_design,
    on="Test_ID",
    how="left",
    suffixes=("_Measurement", "_Design")
)

sinusoidal_row_check_df["Expected_RH_Command"] = (
    sinusoidal_row_check_df["RH_Center"]
    +
    sinusoidal_row_check_df["RH_Amplitude"]
    * np.sin(
        2.0
        * np.pi
        * sinusoidal_row_check_df["Elapsed_s"]
        / sinusoidal_row_check_df["Period_s_Design"]
    )
)

sinusoidal_row_check_df["Command_Matches_Design"] = np.isclose(
    sinusoidal_row_check_df["RH_Command"],
    sinusoidal_row_check_df["Expected_RH_Command"],
    atol=1e-6
)

display(
    sinusoidal_row_check_df[
        [
            "Test_ID",
            "Elapsed_s",
            "RH_Center",
            "RH_Amplitude",
            "RH_Command",
            "Expected_RH_Command",
            "Command_Matches_Design",
        ]
    ].head(15)
)

,Test_ID,Elapsed_s,RH_Center,RH_Amplitude,RH_Command,Expected_RH_Command,Command_Matches_Design
0,DY-SIN-001,0.0,50.0,10.0,50.000000,50.000000,True
1,DY-SIN-001,1.0,50.0,10.0,50.104718,50.104718,True
2,DY-SIN-001,2.0,50.0,10.0,50.209424,50.209424,True
3,DY-SIN-001,3.0,50.0,10.0,50.314108,50.314108,True
4,DY-SIN-001,4.0,50.0,10.0,50.418757,50.418757,True
5,DY-SIN-001,5.0,50.0,10.0,50.523360,50.523360,True
6,DY-SIN-001,6.0,50.0,10.0,50.627905,50.627905,True
7,DY-SIN-001,7.0,50.0,10.0,50.732382,50.732382,True
8,DY-SIN-001,8.0,50.0,10.0,50.836778,50.836778,True
9,DY-SIN-001,9.0,50.0,10.0,50.941083,50.941083,True


In [188]:
# ------------------------------------------------------------
# Summarize sinusoidal command-trajectory integrity
# ------------------------------------------------------------

sinusoidal_command_summary = (
    sinusoidal_row_check_df
    .groupby("Test_ID")
    .agg(
        Samples_Checked=("Sample_ID", "size"),

        Command_Mismatches=(
            "Command_Matches_Design",
            lambda x: (~x).sum()
        ),

        Maximum_Command_Error=(
            "RH_Command",
            lambda x: 0.0
        ),
    )
    .reset_index()
)

# Compute maximum absolute command-model difference separately
command_error_summary = (
    sinusoidal_row_check_df
    .assign(
        Command_Error=lambda df: (
            df["RH_Command"]
            - df["Expected_RH_Command"]
        ).abs()
    )
    .groupby("Test_ID")["Command_Error"]
    .max()
    .reset_index(
        name="Maximum_Command_Error"
    )
)

sinusoidal_command_summary = (
    sinusoidal_command_summary
    .drop(columns=["Maximum_Command_Error"])
    .merge(
        command_error_summary,
        on="Test_ID",
        how="left"
    )
)

display(sinusoidal_command_summary)

,Test_ID,Samples_Checked,Command_Mismatches,Maximum_Command_Error
0,DY-SIN-001,3001,0,4.954530e-07
1,DY-SIN-002,1501,0,4.927131e-07
2,DY-SIN-003,601,0,4.954530e-07
3,DY-SIN-004,301,0,4.770753e-07


In [189]:
# ------------------------------------------------------------
# Final sinusoidal experimental-state qualification
# ------------------------------------------------------------

sinusoidal_command_pass = (
    sinusoidal_command_summary[
        "Command_Mismatches"
    ] == 0
).all()

sinusoidal_metadata_pass = (
    (sinusoidal_state_check_df["_merge"] == "both").all()
    and sinusoidal_state_check_df[
        "Metadata_Internal_Consistency"
    ].all()
    and sinusoidal_state_check_df[
        "Temperature_Matches_Design"
    ].all()
    and sinusoidal_state_check_df[
        "Period_Matches_Design"
    ].all()
    and sinusoidal_state_check_df[
        "Frequency_Matches_Design"
    ].all()
    and sinusoidal_state_check_df[
        "Low_Command_Matches_Design"
    ].all()
    and sinusoidal_state_check_df[
        "High_Command_Matches_Design"
    ].all()
)

dynamic_sinusoidal_state_pass = (
    sinusoidal_metadata_pass
    and sinusoidal_command_pass
)

print(
    f"Sinusoidal tests checked             : "
    f"{len(sinusoidal_state_check_df)}"
)

print(
    f"Internally inconsistent metadata     : "
    f"{(~sinusoidal_state_check_df['Metadata_Internal_Consistency']).sum()}"
)

print(
    f"Temperature mismatches               : "
    f"{(~sinusoidal_state_check_df['Temperature_Matches_Design']).sum()}"
)

print(
    f"Period mismatches                    : "
    f"{(~sinusoidal_state_check_df['Period_Matches_Design']).sum()}"
)

print(
    f"Frequency mismatches                 : "
    f"{(~sinusoidal_state_check_df['Frequency_Matches_Design']).sum()}"
)

print(
    f"Low-command-limit mismatches         : "
    f"{(~sinusoidal_state_check_df['Low_Command_Matches_Design']).sum()}"
)

print(
    f"High-command-limit mismatches        : "
    f"{(~sinusoidal_state_check_df['High_Command_Matches_Design']).sum()}"
)

print(
    f"Command-trajectory mismatches        : "
    f"{sinusoidal_command_summary['Command_Mismatches'].sum()}"
)

print(
    f"\nSinusoidal state qualification       : "
    f"{'PASS' if dynamic_sinusoidal_state_pass else 'FAIL'}"
)

Sinusoidal tests checked             : 4
Internally inconsistent metadata     : 0
Temperature mismatches               : 0
Period mismatches                    : 0
Frequency mismatches                 : 0
Low-command-limit mismatches         : 0
High-command-limit mismatches        : 0
Command-trajectory mismatches        : 0

Sinusoidal state qualification       : PASS


### 7.7 Stability Experimental-State Consistency

The stability schedule, aging day, RH setpoint, and temperature setpoint were
qualified previously in Section 6.4.

This subsection completes the experimental-state assessment by verifying that
measurement-state metadata remain internally consistent within each stability
test.

Because the experimental test matrix does not explicitly prescribe
`Direction` for stability tests, direction is treated as observational
metadata rather than as a design acceptance criterion.

The chamber state is expected to remain `STABLE` throughout stability
measurements.

Sensor drift magnitude is not evaluated here.

In [190]:
# ------------------------------------------------------------
# Summarize additional stability-state metadata
# ------------------------------------------------------------

stability_state_extra = (
    stability_df
    .groupby("Test_ID")
    .agg(
        Observed_Direction=("Direction", "first"),
        Unique_Directions=("Direction", "nunique"),

        Observed_Chamber_State=("Chamber_State", "first"),
        Unique_Chamber_States=("Chamber_State", "nunique"),
    )
    .reset_index()
)

display(stability_state_extra)

,Test_ID,Observed_Direction,Unique_Directions,Observed_Chamber_State,Unique_Chamber_States
0,STAB-001,UP,1,STABLE,1
1,STAB-002,UP,1,STABLE,1
2,STAB-003,UP,1,STABLE,1
3,STAB-004,UP,1,STABLE,1
4,STAB-005,UP,1,STABLE,1
5,STAB-006,UP,1,STABLE,1
6,STAB-007,UP,1,STABLE,1
7,STAB-008,UP,1,STABLE,1
8,STAB-009,UP,1,STABLE,1
9,STAB-010,UP,1,STABLE,1


In [191]:
# ------------------------------------------------------------
# Evaluate stability state consistency
# ------------------------------------------------------------

stability_state_extra["Direction_Internal_Consistency"] = (
    stability_state_extra["Unique_Directions"] == 1
)

stability_state_extra["Chamber_State_Internal_Consistency"] = (
    stability_state_extra["Unique_Chamber_States"] == 1
)

stability_state_extra["Chamber_State_Is_Stable"] = (
    stability_state_extra["Observed_Chamber_State"]
    .astype(str)
    .str.strip()
    .str.upper()
    == "STABLE"
)

display(
    stability_state_extra[
        [
            "Test_ID",
            "Observed_Direction",
            "Unique_Directions",
            "Observed_Chamber_State",
            "Unique_Chamber_States",
            "Direction_Internal_Consistency",
            "Chamber_State_Internal_Consistency",
            "Chamber_State_Is_Stable",
        ]
    ]
)

,Test_ID,Observed_Direction,Unique_Directions,Observed_Chamber_State,Unique_Chamber_States,Direction_Internal_Consistency,Chamber_State_Internal_Consistency,Chamber_State_Is_Stable
0,STAB-001,UP,1,STABLE,1,True,True,True
1,STAB-002,UP,1,STABLE,1,True,True,True
2,STAB-003,UP,1,STABLE,1,True,True,True
3,STAB-004,UP,1,STABLE,1,True,True,True
4,STAB-005,UP,1,STABLE,1,True,True,True
5,STAB-006,UP,1,STABLE,1,True,True,True
6,STAB-007,UP,1,STABLE,1,True,True,True
7,STAB-008,UP,1,STABLE,1,True,True,True
8,STAB-009,UP,1,STABLE,1,True,True,True
9,STAB-010,UP,1,STABLE,1,True,True,True


In [192]:
# ------------------------------------------------------------
# Combine schedule integrity with stability-state integrity
# ------------------------------------------------------------

stability_state_check_df = stability_check_df.merge(
    stability_state_extra,
    on="Test_ID",
    how="left"
)

display(
    stability_state_check_df[
        [
            "Test_ID",
            "Schedule_Type",
            "Aging_Day",
            "RH_Setpoint",
            "Temperature_Setpoint_C",
            "Observed_Direction",
            "Observed_Chamber_State",
        ]
    ]
)

,Test_ID,Schedule_Type,Aging_Day,RH_Setpoint,Temperature_Setpoint_C,Observed_Direction,Observed_Chamber_State
0,STAB-001,WEEKLY_CENTER_CHECK,0.0,50.0,25.0,UP,STABLE
1,STAB-002,WEEKLY_CENTER_CHECK,7.0,50.0,25.0,UP,STABLE
2,STAB-003,WEEKLY_CENTER_CHECK,14.0,50.0,25.0,UP,STABLE
3,STAB-004,WEEKLY_CENTER_CHECK,21.0,50.0,25.0,UP,STABLE
4,STAB-005,WEEKLY_CENTER_CHECK,28.0,50.0,25.0,UP,STABLE
5,STAB-006,WEEKLY_CENTER_CHECK,35.0,50.0,25.0,UP,STABLE
6,STAB-007,WEEKLY_CENTER_CHECK,42.0,50.0,25.0,UP,STABLE
7,STAB-008,WEEKLY_CENTER_CHECK,49.0,50.0,25.0,UP,STABLE
8,STAB-009,WEEKLY_CENTER_CHECK,56.0,50.0,25.0,UP,STABLE
9,STAB-010,WEEKLY_CENTER_CHECK,63.0,50.0,25.0,UP,STABLE


In [193]:
# ------------------------------------------------------------
# Final stability experimental-state qualification
# ------------------------------------------------------------

stability_state_pass = (
    stability_integrity_pass
    and stability_state_extra[
        "Direction_Internal_Consistency"
    ].all()
    and stability_state_extra[
        "Chamber_State_Internal_Consistency"
    ].all()
    and stability_state_extra[
        "Chamber_State_Is_Stable"
    ].all()
)

print(
    f"Stability tests checked                 : "
    f"{len(stability_state_extra)}"
)

print(
    f"Tests with varying Direction            : "
    f"{(~stability_state_extra['Direction_Internal_Consistency']).sum()}"
)

print(
    f"Tests with varying Chamber_State        : "
    f"{(~stability_state_extra['Chamber_State_Internal_Consistency']).sum()}"
)

print(
    f"Tests not recorded as STABLE            : "
    f"{(~stability_state_extra['Chamber_State_Is_Stable']).sum()}"
)

print(
    f"Schedule/design integrity               : "
    f"{'PASS' if stability_integrity_pass else 'FAIL'}"
)

print(
    f"\nStability experimental-state qualification : "
    f"{'PASS' if stability_state_pass else 'FAIL'}"
)

Stability tests checked                 : 22
Tests with varying Direction            : 0
Tests with varying Chamber_State        : 0
Tests not recorded as STABLE            : 0
Schedule/design integrity               : PASS

Stability experimental-state qualification : PASS


In [194]:
# ------------------------------------------------------------
# Final Section 7 experimental-state summary
# ------------------------------------------------------------

experimental_state_summary = pd.DataFrame(
    [
        {
            "Qualification_Check": "Test-family mapping",
            "Status": "PASS" if test_family_mapping_pass else "FAIL",
        },
        {
            "Qualification_Check": "Static experimental state",
            "Status": "PASS" if static_state_pass else "FAIL",
        },
        {
            "Qualification_Check": "Dynamic-step experimental state",
            "Status": "PASS" if dynamic_step_state_pass else "FAIL",
        },
        {
            "Qualification_Check": "Dynamic-ramp experimental state",
            "Status": "PASS" if dynamic_ramp_state_pass else "FAIL",
        },
        {
            "Qualification_Check": "Dynamic-cyclic experimental state",
            "Status": "PASS" if dynamic_cyclic_state_pass else "FAIL",
        },
        {
            "Qualification_Check": "Dynamic-sinusoidal experimental state",
            "Status": "PASS" if dynamic_sinusoidal_state_pass else "FAIL",
        },
        {
            "Qualification_Check": "Stability experimental state",
            "Status": "PASS" if stability_state_pass else "FAIL",
        },
    ]
)

display(experimental_state_summary)

experimental_state_pass = (
    experimental_state_summary["Status"] == "PASS"
).all()

print(
    f"\nExperimental-state consistency : "
    f"{'PASS' if experimental_state_pass else 'FAIL'}"
)

,Qualification_Check,Status
0,Test-family mapping,PASS
1,Static experimental state,PASS
2,Dynamic-step experimental state,PASS
3,Dynamic-ramp experimental state,PASS
4,Dynamic-cyclic experimental state,PASS
5,Dynamic-sinusoidal experimental state,PASS
6,Stability experimental state,PASS



Experimental-state consistency : PASS


## 8. Documented Design Conditions and Exceptions

Dataset qualification must distinguish between true data defects and
intentional characteristics of the experimental design.

Several structural conditions in the frozen release could appear anomalous
if evaluated using generic data-cleaning rules. These conditions are therefore
documented explicitly and assessed according to their engineering meaning.

A documented design condition is not treated as a qualification failure when:

1. it is consistent with the controlled experimental design,
2. it does not break measurement traceability,
3. an appropriate family-specific qualification method is available, and
4. it does not compromise the intended downstream analysis.

In [195]:
# ------------------------------------------------------------
# Document matrix-only uncertainty-evaluation design points
# ------------------------------------------------------------

matrix_only_test_ids = sorted(
    matrix_test_ids - measurement_test_ids
)

matrix_only_design_points = test_matrix_df[
    test_matrix_df["Test_ID"].isin(matrix_only_test_ids)
][
    [
        "Test_ID",
        "Test_Family",
        "RH_Setpoint",
        "Temperature_Setpoint_C",
        "Point_Type",
    ]
].copy()

display(matrix_only_design_points)

uncertainty_design_only_pass = (
    len(matrix_only_design_points) == 4
    and (
        matrix_only_design_points["Test_Family"]
        == "UNCERTAINTY_EVALUATION"
    ).all()
)

print(
    f"Matrix-only Test_IDs                    : "
    f"{len(matrix_only_design_points)}"
)

print(
    f"All are uncertainty-evaluation points   : "
    f"{uncertainty_design_only_pass}"
)

,Test_ID,Test_Family,RH_Setpoint,Temperature_Setpoint_C,Point_Type
329,UNC-001,UNCERTAINTY_EVALUATION,30.0,25.0,REPRESENTATIVE
330,UNC-002,UNCERTAINTY_EVALUATION,50.0,25.0,REPRESENTATIVE
331,UNC-003,UNCERTAINTY_EVALUATION,70.0,25.0,REPRESENTATIVE
332,UNC-004,UNCERTAINTY_EVALUATION,80.0,35.0,WORST_CASE


Matrix-only Test_IDs                    : 4
All are uncertainty-evaluation points   : True


In [196]:
# ------------------------------------------------------------
# Document family-specific sample-count definitions
# ------------------------------------------------------------

expected_sample_design_summary = (
    test_matrix_df
    .groupby("Test_Family")
    .agg(
        Tests=("Test_ID", "size"),
        Explicit_Expected_Samples=(
            "Expected_Samples",
            lambda x: x.notna().sum()
        ),
        Defined_Duration=(
            "Duration_s",
            lambda x: x.notna().sum()
        ),
        Defined_Sample_Rate=(
            "Sample_Rate_Hz",
            lambda x: x.notna().sum()
        ),
    )
    .reset_index()
)

display(expected_sample_design_summary)

,Test_Family,Tests,Explicit_Expected_Samples,Defined_Duration,Defined_Sample_Rate
0,DYNAMIC_CYCLIC,3,0,0,3
1,DYNAMIC_RAMP,18,0,18,18
2,DYNAMIC_SINUSOIDAL,4,0,0,4
3,DYNAMIC_STEP,30,0,30,30
4,STABILITY,22,0,0,0
5,STATIC,252,252,0,252
6,UNCERTAINTY_EVALUATION,4,0,0,0


In [197]:
sample_count_design_condition_pass = (
    explicit_count_pass
    and cyclic_coverage_pass
    and sinusoidal_integrity_pass
    and stability_integrity_pass
)

print(
    f"Family-aware completeness qualification : "
    f"{'PASS' if sample_count_design_condition_pass else 'FAIL'}"
)

Family-aware completeness qualification : PASS


In [198]:
# ------------------------------------------------------------
# Document dynamic-step chamber-state representation
# ------------------------------------------------------------

step_chamber_states = sorted(
    step_df["Chamber_State"]
    .dropna()
    .astype(str)
    .str.strip()
    .str.upper()
    .unique()
)

step_design_requires_chamber_state = (
    test_matrix_df.loc[
        test_matrix_df["Test_Family"] == "DYNAMIC_STEP",
        "Chamber_State_Required"
    ]
    .notna()
    .any()
)

print(
    f"Observed dynamic-step chamber states : "
    f"{step_chamber_states}"
)

print(
    f"Explicit chamber-state requirement in matrix : "
    f"{step_design_requires_chamber_state}"
)

print(
    f"Step segment qualification : "
    f"{'PASS' if step_chamber_state_pass else 'FAIL'}"
)

Observed dynamic-step chamber states : ['STABLE', 'TRANSIENT']
Explicit chamber-state requirement in matrix : False
Step segment qualification : PASS


In [199]:
# ------------------------------------------------------------
# Formal design-condition / exception register
# ------------------------------------------------------------

design_condition_register = pd.DataFrame(
    [
        {
            "ID": "DC-001",
            "Condition": (
                "Four UNCERTAINTY_EVALUATION Test_IDs exist in "
                "the test matrix without standalone measurement rows."
            ),
            "Engineering_Interpretation": (
                "These are design-only uncertainty evaluation points."
            ),
            "Qualification_Impact": "NON-BLOCKING",
            "Status": (
                "ACCEPTED"
                if uncertainty_design_only_pass
                else "REVIEW"
            ),
        },
        {
            "ID": "DC-002",
            "Condition": (
                "Expected_Samples is not explicitly defined for "
                "all test families."
            ),
            "Engineering_Interpretation": (
                "Completeness is qualified using family-specific "
                "experimental-design criteria."
            ),
            "Qualification_Impact": "NON-BLOCKING",
            "Status": (
                "ACCEPTED"
                if sample_count_design_condition_pass
                else "REVIEW"
            ),
        },
        {
            "ID": "DC-003",
            "Condition": (
                "Dynamic-step records use STABLE baseline and "
                "TRANSIENT post-step states without a separate "
                "SETTLING label."
            ),
            "Engineering_Interpretation": (
                "Settling behavior will be derived from the "
                "measured response during dynamic characterization."
            ),
            "Qualification_Impact": "NON-BLOCKING",
            "Status": (
                "ACCEPTED"
                if step_chamber_state_pass
                else "REVIEW"
            ),
        },
    ]
)

display(design_condition_register)

design_conditions_accepted = (
    design_condition_register["Status"] == "ACCEPTED"
).all()

print(
    f"\nDocumented design conditions : "
    f"{'ACCEPTED' if design_conditions_accepted else 'REVIEW REQUIRED'}"
)

,ID,Condition,Engineering_Interpretation,Qualification_Impact,Status
0,DC-001,Four UNCERTAINTY_EVALUATION Test_IDs exist in ...,These are design-only uncertainty evaluation p...,NON-BLOCKING,ACCEPTED
1,DC-002,Expected_Samples is not explicitly defined for...,Completeness is qualified using family-specifi...,NON-BLOCKING,ACCEPTED
2,DC-003,Dynamic-step records use STABLE baseline and T...,Settling behavior will be derived from the mea...,NON-BLOCKING,ACCEPTED



Documented design conditions : ACCEPTED


## 9. Final DQ-001 Qualification Decision

The purpose of DQ-001 is to determine whether the frozen analyst-facing
dataset is structurally complete, internally consistent, traceable to the
experimental design, and suitable for downstream sensor-validation analysis.

The final decision integrates all qualification evidence developed in this
notebook.

Documented non-blocking design conditions are retained explicitly rather than
being treated as data defects.

In [200]:
# ------------------------------------------------------------
# Build final DQ-001 qualification matrix
# ------------------------------------------------------------

dq001_qualification_matrix = pd.DataFrame(
    [
        {
            "DQ_Check": "Dataset inventory",
            "Status": inventory_status,
        },
        {
            "DQ_Check": "Schema integrity",
            "Status": "PASS" if schema_pass else "FAIL",
        },
        {
            "DQ_Check": "Row-count integrity",
            "Status": "PASS" if row_count_pass else "FAIL",
        },
        {
            "DQ_Check": "Required-field completeness",
            "Status": "PASS" if missing_value_pass else "FAIL",
        },
        {
            "DQ_Check": "Duplicate integrity",
            "Status": "PASS" if duplicate_integrity_pass else "FAIL",
        },
        {
            "DQ_Check": "Test-ID traceability",
            "Status": "PASS" if test_id_traceability_pass else "FAIL",
        },
        {
            "DQ_Check": "Family-aware test completeness",
            "Status": (
                "PASS"
                if sample_count_design_condition_pass
                else "FAIL"
            ),
        },
        {
            "DQ_Check": "Calibration / validation role mapping",
            "Status": "PASS" if role_mapping_pass else "FAIL",
        },
        {
            "DQ_Check": "Analysis-role isolation",
            "Status": "PASS" if role_isolation_pass else "FAIL",
        },
        {
            "DQ_Check": "Basic time-sequence integrity",
            "Status": "PASS" if time_sequence_pass else "FAIL",
        },
        {
            "DQ_Check": "Sampling-interval consistency",
            "Status": (
                "PASS"
                if sampling_interval_pass
                else "FAIL"
            ),
        },
        {
            "DQ_Check": "Experimental-state consistency",
            "Status": (
                "PASS"
                if experimental_state_pass
                else "FAIL"
            ),
        },
        {
            "DQ_Check": "Documented design conditions",
            "Status": (
                "ACCEPTED"
                if design_conditions_accepted
                else "REVIEW"
            ),
        },
    ]
)

display(dq001_qualification_matrix)

,DQ_Check,Status
0,Dataset inventory,PASS
1,Schema integrity,PASS
2,Row-count integrity,PASS
3,Required-field completeness,PASS
4,Duplicate integrity,PASS
5,Test-ID traceability,PASS
6,Family-aware test completeness,PASS
7,Calibration / validation role mapping,PASS
8,Analysis-role isolation,PASS
9,Basic time-sequence integrity,PASS


In [201]:
# ------------------------------------------------------------
# Formal DQ-001 decision
# ------------------------------------------------------------

blocking_checks = dq001_qualification_matrix[
    dq001_qualification_matrix["DQ_Check"]
    != "Documented design conditions"
]

all_blocking_checks_pass = (
    blocking_checks["Status"] == "PASS"
).all()

if not all_blocking_checks_pass:
    dq001_decision = "FAIL"

elif design_conditions_accepted:
    dq001_decision = "PASS WITH DOCUMENTED DESIGN CONDITIONS"

else:
    dq001_decision = "REVIEW REQUIRED"

print("=" * 60)
print("DQ-001 DATASET QUALIFICATION")
print("=" * 60)

print(
    f"Blocking qualification checks passed : "
    f"{all_blocking_checks_pass}"
)

print(
    f"Documented design conditions accepted: "
    f"{design_conditions_accepted}"
)

print(
    f"\nFINAL DQ-001 DECISION: {dq001_decision}"
)

print("=" * 60)

DQ-001 DATASET QUALIFICATION
Blocking qualification checks passed : True
Documented design conditions accepted: True

FINAL DQ-001 DECISION: PASS WITH DOCUMENTED DESIGN CONDITIONS


In [202]:
# ------------------------------------------------------------
# Diagnose blocking DQ-001 failures
# ------------------------------------------------------------

display(
    dq001_qualification_matrix[
        dq001_qualification_matrix["Status"] != "PASS"
    ]
)

,DQ_Check,Status
12,Documented design conditions,ACCEPTED


In [203]:
blocking_failures = blocking_checks[
    blocking_checks["Status"] != "PASS"
]

display(blocking_failures)

print(f"Number of blocking failures: {len(blocking_failures)}")

,DQ_Check,Status


Number of blocking failures: 0


In [204]:
# ------------------------------------------------------------
# Inspect individual qualification flags
# ------------------------------------------------------------

qualification_flags = {
    "schema_pass": schema_pass,
    "row_count_pass": row_count_pass,
    "missing_value_pass": missing_value_pass,
    "duplicate_integrity_pass": duplicate_integrity_pass,
    "test_id_traceability_pass": test_id_traceability_pass,
    "sample_count_design_condition_pass": sample_count_design_condition_pass,
    "role_mapping_pass": role_mapping_pass,
    "role_isolation_pass": role_isolation_pass,
    "time_sequence_pass": time_sequence_pass,
    "sampling_interval_pass": sampling_interval_pass,
    "experimental_state_pass": experimental_state_pass,
    "design_conditions_accepted": design_conditions_accepted,
}

for name, value in qualification_flags.items():
    print(f"{name:<38} : {value}")

schema_pass                            : True
row_count_pass                         : True
missing_value_pass                     : True
duplicate_integrity_pass               : True
test_id_traceability_pass              : True
sample_count_design_condition_pass     : True
role_mapping_pass                      : True
role_isolation_pass                    : True
time_sequence_pass                     : True
sampling_interval_pass                 : True
experimental_state_pass                : True
design_conditions_accepted             : True


In [205]:
experimental_state_pass = all(
    [
        test_family_mapping_pass,
        static_state_pass,
        dynamic_step_state_pass,
        dynamic_ramp_state_pass,
        dynamic_cyclic_state_pass,
        dynamic_sinusoidal_state_pass,
        stability_state_pass,
    ]
)

print(
    "Experimental-state consistency:",
    "PASS" if experimental_state_pass else "FAIL"
)

Experimental-state consistency: PASS


In [206]:
# ------------------------------------------------------------
# Diagnose Section 7 sub-checks
# ------------------------------------------------------------

experimental_state_flags = {
    "7.1 Test-family mapping":
        test_family_mapping_pass,

    "7.2 Static experimental state":
        static_state_pass,

    "7.3 Dynamic-step experimental state":
        dynamic_step_state_pass,

    "7.4 Dynamic-ramp experimental state":
        dynamic_ramp_state_pass,

    "7.5 Dynamic-cyclic experimental state":
        dynamic_cyclic_state_pass,

    "7.6 Dynamic-sinusoidal experimental state":
        dynamic_sinusoidal_state_pass,

    "7.7 Stability experimental state":
        stability_state_pass,
}

for check_name, status in experimental_state_flags.items():
    print(
        f"{check_name:<45} : "
        f"{'PASS' if bool(status) else 'FAIL'}"
    )

7.1 Test-family mapping                       : PASS
7.2 Static experimental state                 : PASS
7.3 Dynamic-step experimental state           : PASS
7.4 Dynamic-ramp experimental state           : PASS
7.5 Dynamic-cyclic experimental state         : PASS
7.6 Dynamic-sinusoidal experimental state     : PASS
7.7 Stability experimental state              : PASS


In [207]:
# ------------------------------------------------------------
# Diagnose 7.1 Test-family mapping failure
# ------------------------------------------------------------

print(
    f"Tests with multiple family labels  : "
    f"{len(multi_family_measurement_tests)}"
)

print(
    f"Measurement/matrix family mismatch : "
    f"{len(family_mismatches)}"
)

print(
    f"Untraceable measurement families   : "
    f"{len(untraceable_family_tests)}"
)

Tests with multiple family labels  : 0
Measurement/matrix family mismatch : 0
Untraceable measurement families   : 0


In [208]:
if len(multi_family_measurement_tests) > 0:
    print("Tests with multiple family labels:")
    display(multi_family_measurement_tests)

In [209]:
if len(family_mismatches) > 0:
    print("Measurement / matrix family mismatches:")
    display(
        family_mismatches[
            [
                "Test_ID",
                "Source_File",
                "Measurement_Test_Family",
                "Matrix_Test_Family",
                "_merge",
            ]
        ]
    )

In [210]:
if len(untraceable_family_tests) > 0:
    print("Untraceable measurement-family records:")
    display(
        untraceable_family_tests[
            [
                "Test_ID",
                "Source_File",
                "Measurement_Test_Family",
                "Matrix_Test_Family",
                "_merge",
            ]
        ]
    )

In [211]:
experimental_state_flags = {
    "7.1 Test-family mapping":
        test_family_mapping_pass,

    "7.2 Static experimental state":
        static_state_pass,

    "7.3 Dynamic-step experimental state":
        dynamic_step_state_pass,

    "7.4 Dynamic-ramp experimental state":
        dynamic_ramp_state_pass,

    "7.5 Dynamic-cyclic experimental state":
        dynamic_cyclic_state_pass,

    "7.6 Dynamic-sinusoidal experimental state":
        dynamic_sinusoidal_state_pass,

    "7.7 Stability experimental state":
        stability_state_pass,
}

for check_name, status in experimental_state_flags.items():
    print(
        f"{check_name:<45} : "
        f"{'PASS' if bool(status) else 'FAIL'}"
    )

experimental_state_pass = all(
    bool(status)
    for status in experimental_state_flags.values()
)

print(
    f"\nExperimental-state consistency : "
    f"{'PASS' if experimental_state_pass else 'FAIL'}"
)

7.1 Test-family mapping                       : PASS
7.2 Static experimental state                 : PASS
7.3 Dynamic-step experimental state           : PASS
7.4 Dynamic-ramp experimental state           : PASS
7.5 Dynamic-cyclic experimental state         : PASS
7.6 Dynamic-sinusoidal experimental state     : PASS
7.7 Stability experimental state              : PASS

Experimental-state consistency : PASS


In [212]:
dq001_qualification_matrix = pd.DataFrame(
    [
        {
            "DQ_Check": "Dataset inventory",
            "Status": inventory_status,
        },
        {
            "DQ_Check": "Schema integrity",
            "Status": "PASS" if schema_pass else "FAIL",
        },
        {
            "DQ_Check": "Row-count integrity",
            "Status": "PASS" if row_count_pass else "FAIL",
        },
        {
            "DQ_Check": "Required-field completeness",
            "Status": "PASS" if missing_value_pass else "FAIL",
        },
        {
            "DQ_Check": "Duplicate integrity",
            "Status": "PASS" if duplicate_integrity_pass else "FAIL",
        },
        {
            "DQ_Check": "Test-ID traceability",
            "Status": "PASS" if test_id_traceability_pass else "FAIL",
        },
        {
            "DQ_Check": "Family-aware test completeness",
            "Status": "PASS" if sample_count_design_condition_pass else "FAIL",
        },
        {
            "DQ_Check": "Calibration / validation role mapping",
            "Status": "PASS" if role_mapping_pass else "FAIL",
        },
        {
            "DQ_Check": "Analysis-role isolation",
            "Status": "PASS" if role_isolation_pass else "FAIL",
        },
        {
            "DQ_Check": "Basic time-sequence integrity",
            "Status": "PASS" if time_sequence_pass else "FAIL",
        },
        {
            "DQ_Check": "Sampling-interval consistency",
            "Status": "PASS" if sampling_interval_pass else "FAIL",
        },
        {
            "DQ_Check": "Experimental-state consistency",
            "Status": "PASS" if experimental_state_pass else "FAIL",
        },
        {
            "DQ_Check": "Documented design conditions",
            "Status": (
                "ACCEPTED"
                if design_conditions_accepted
                else "REVIEW"
            ),
        },
    ]
)

display(dq001_qualification_matrix)

,DQ_Check,Status
0,Dataset inventory,PASS
1,Schema integrity,PASS
2,Row-count integrity,PASS
3,Required-field completeness,PASS
4,Duplicate integrity,PASS
5,Test-ID traceability,PASS
6,Family-aware test completeness,PASS
7,Calibration / validation role mapping,PASS
8,Analysis-role isolation,PASS
9,Basic time-sequence integrity,PASS


In [213]:
blocking_checks = dq001_qualification_matrix[
    dq001_qualification_matrix["DQ_Check"]
    != "Documented design conditions"
]

all_blocking_checks_pass = (
    blocking_checks["Status"] == "PASS"
).all()

if not all_blocking_checks_pass:
    dq001_decision = "FAIL"

elif design_conditions_accepted:
    dq001_decision = "PASS WITH DOCUMENTED DESIGN CONDITIONS"

else:
    dq001_decision = "REVIEW REQUIRED"

print("=" * 60)
print("DQ-001 DATASET QUALIFICATION")
print("=" * 60)

print(
    f"Blocking qualification checks passed : "
    f"{all_blocking_checks_pass}"
)

print(
    f"Documented design conditions accepted: "
    f"{design_conditions_accepted}"
)

print(
    f"\nFINAL DQ-001 DECISION: {dq001_decision}"
)

print("=" * 60)

DQ-001 DATASET QUALIFICATION
Blocking qualification checks passed : True
Documented design conditions accepted: True

FINAL DQ-001 DECISION: PASS WITH DOCUMENTED DESIGN CONDITIONS


## Engineering Interpretation

The frozen analyst-facing dataset successfully passed all blocking DQ-001
qualification checks.

The qualification demonstrated that the dataset is:

- structurally complete
- consistent with the frozen release manifest
- free of missing values in required measurement fields
- free of duplicate measurement identities
- traceable from measurement records to the controlled test matrix
- complete according to family-specific experimental-design criteria
- correctly partitioned between calibration, validation, and characterization
- temporally ordered and internally consistent
- consistent with the commanded experimental states
- suitable for downstream sensor-validation analysis

Three non-blocking design conditions were identified and formally documented:

1. Four `UNCERTAINTY_EVALUATION` entries exist as design-only points without
   standalone measurement records.
2. `Expected_Samples` is not explicitly defined for every test family, so
   completeness must be evaluated using family-specific criteria.
3. Dynamic-step tests use `STABLE` baseline and `TRANSIENT` post-step labels
   without a separate `SETTLING` state.

These conditions do not compromise measurement traceability or the intended
downstream analysis.

### DQ-001 Decision

**PASS WITH DOCUMENTED DESIGN CONDITIONS**

The dataset is qualified for subsequent sensor characterization, calibration,
independent validation, dynamic analysis, stability assessment, and
measurement-uncertainty evaluation.

## What I Learned

Dataset qualification is more than generic data cleaning.

A measurement dataset may contain no missing values and still be unsuitable
for engineering analysis if the measurements cannot be traced to the
experimental design or if the experimental conditions are inconsistent.

The most important lesson from DQ-001 is that qualification criteria must be
derived from the measurement process and test design.

Different test families require different completeness rules:

- static tests can use an explicit expected sample count
- step and ramp tests can use duration and sampling-rate information
- cyclic tests require cycle-coverage checks
- sinusoidal tests require waveform and period-frequency consistency
- stability tests require aging-schedule consistency

Therefore, engineering context determines how data-quality rules should be
constructed.

## Key Metrology Concepts

### Measurement Traceability

Each measurement record must be traceable to a defined experimental condition
before it can support an engineering conclusion.

### Experimental-Design Integrity

Measurement metadata must agree with the conditions specified by the test
design, including setpoints, direction, repetitions, sessions, timing, and
dynamic excitation parameters.

### Calibration / Validation Separation

Calibration observations are used to develop a correction model.

Validation observations must remain independent until the calibration model
has been frozen. Reusing calibration observations for validation would produce
an optimistically biased assessment of model performance.

### Family-Aware Qualification

Different experiments represent completeness differently. A single generic
rule cannot be applied safely to static, step, ramp, cyclic, sinusoidal, and
stability measurements.

### Data Quality vs. Sensor Performance

DQ-001 determines whether the measurement evidence is trustworthy.

It does not determine whether the DUT itself performs well.

A sensor may later fail an accuracy, repeatability, response-time, drift, or
uncertainty requirement even when the dataset itself is fully qualified.

## Questions I Should Be Able to Answer

After completing DQ-001, I should be able to explain:

1. Why is dataset qualification required before sensor-performance analysis?

2. What is the difference between file-level completeness and test-level
   completeness?

3. Why can a missing `Expected_Samples` value be valid rather than a data
   defect?

4. Why must calibration and validation datasets remain independent?

5. Why is monotonic elapsed time not sufficient to prove correct sampling?

6. How can test-matrix metadata be used to validate measurement records?

7. Why must dynamic test families use different qualification logic from
   static tests?

8. What is the difference between verifying the commanded excitation and
   evaluating the DUT response?

9. Why are the four `UNCERTAINTY_EVALUATION` matrix-only points not considered
   missing experiments?

10. What does `PASS WITH DOCUMENTED DESIGN CONDITIONS` mean in an engineering
    qualification process?

## Connection to the Uncertainty Budget

Dataset qualification provides the evidence foundation required for the
measurement-uncertainty analysis performed later in the project.

An uncertainty budget is meaningful only if the measurements used to estimate
repeatability, calibration residuals, drift, environmental effects, and other
uncertainty contributions are structurally valid and traceable to known test
conditions.

DQ-001 therefore acts as a prerequisite gate for the uncertainty analysis.

The four `UNCERTAINTY_EVALUATION` design points will later identify
representative and worst-case operating conditions at which the combined and
expanded measurement uncertainty can be evaluated.

## Final Engineering Takeaway

The analyst-facing HVAC relative-humidity sensor dataset is sufficiently
complete, traceable, internally consistent, and experimentally coherent for
downstream validation analysis.

No blocking dataset-integrity defect was identified.

The identified structural differences between test families are intentional
features of the experimental design and have been addressed using
family-specific qualification criteria.

Therefore:

**DQ-001 = PASS WITH DOCUMENTED DESIGN CONDITIONS**

The next engineering stage is static sensor characterization, where the focus
changes from:

> Can the measurement evidence be trusted?

to:

> What does the qualified evidence reveal about DUT measurement performance?

## Interview / Technical Discussion Notes

A useful way to describe this work in a technical discussion is:

"I first qualified the measurement dataset before evaluating sensor
performance. I verified release integrity, schema, missing values, duplicate
records, Test-ID traceability, calibration-validation separation, temporal
integrity, and consistency between measurement metadata and the experimental
test matrix.

Because the campaign contained static, step, ramp, cyclic, sinusoidal, and
stability experiments, I used family-specific qualification criteria rather
than applying generic data-cleaning rules.

Only after the dataset passed the qualification gate did I allow it to proceed
to sensor characterization and calibration."

In [214]:
# ------------------------------------------------------------
# DQ-001 Gate
# ------------------------------------------------------------

assert all_blocking_checks_pass, (
    "DQ-001 cannot be closed because a blocking "
    "qualification check has failed."
)

assert design_conditions_accepted, (
    "DQ-001 cannot be closed because one or more "
    "documented design conditions require review."
)

print("DQ-001 GATE: PASS")
print("Dataset released for downstream analysis.")
print(
    "Next stage: Notebook 01 - Static Characterization"
)

DQ-001 GATE: PASS
Dataset released for downstream analysis.
Next stage: Notebook 01 - Static Characterization
